In [430]:
# Import dependences
import pandas as pd
from sklearn.model_selection import train_test_split
from scipy.stats import f_oneway, chi2_contingency
import math
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
pd.set_option('display.max_rows', None)

In [2]:
# Read the Dataset
dataset = pd.read_csv("train.csv")
dataset

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000
5,6,50,RL,85.0,14115,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,MnPrv,Shed,700,10,2009,WD,Normal,143000
6,7,20,RL,75.0,10084,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2007,WD,Normal,307000
7,8,60,RL,NaN,10382,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,Shed,350,11,2009,WD,Normal,200000
8,9,50,RM,51.0,6120,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2008,WD,Abnorml,129900
9,10,190,RL,50.0,7420,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,1,2008,WD,Normal,118000


In [3]:
# View dataset statistics
dataset.describe(include='all')

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
count,1460.000000,1460.000000,1460,1201.000000,1460.000000,1460,91,1460,1460,1460,...,1460.000000,7,281,54,1460.000000,1460.000000,1460.000000,1460,1460,1460.000000
unique,NaN,NaN,5,NaN,NaN,2,2,4,4,2,...,NaN,3,4,4,NaN,NaN,NaN,9,6,NaN
top,NaN,NaN,RL,NaN,NaN,Pave,Grvl,Reg,Lvl,AllPub,...,NaN,Gd,MnPrv,Shed,NaN,NaN,NaN,WD,Normal,NaN
freq,NaN,NaN,1151,NaN,NaN,1454,50,925,1311,1459,...,NaN,3,157,49,NaN,NaN,NaN,1267,1198,NaN
mean,730.500000,56.897260,NaN,70.049958,10516.828082,NaN,NaN,NaN,NaN,NaN,...,2.758904,NaN,NaN,NaN,43.489041,6.321918,2007.815753,NaN,NaN,180921.195890
std,421.610009,42.300571,NaN,24.284752,9981.264932,NaN,NaN,NaN,NaN,NaN,...,40.177307,NaN,NaN,NaN,496.123024,2.703626,1.328095,NaN,NaN,79442.502883
min,1.000000,20.000000,NaN,21.000000,1300.000000,NaN,NaN,NaN,NaN,NaN,...,0.000000,NaN,NaN,NaN,0.000000,1.000000,2006.000000,NaN,NaN,34900.000000
25%,365.750000,20.000000,NaN,59.000000,7553.500000,NaN,NaN,NaN,NaN,NaN,...,0.000000,NaN,NaN,NaN,0.000000,5.000000,2007.000000,NaN,NaN,129975.000000
50%,730.500000,50.000000,NaN,69.000000,9478.500000,NaN,NaN,NaN,NaN,NaN,...,0.000000,NaN,NaN,NaN,0.000000,6.000000,2008.000000,NaN,NaN,163000.000000
75%,1095.250000,70.000000,NaN,80.000000,11601.500000,NaN,NaN,NaN,NaN,NaN,...,0.000000,NaN,NaN,NaN,0.000000,8.000000,2009.000000,NaN,NaN,214000.000000


In [4]:
# Checking missing values
missing_values = dataset.isnull().sum().sum()
print(f"Are any features missing data? Answer:", "No" if missing_values == 0 else "Yes")

Are any features missing data? Answer: Yes


In [5]:
# How to get column names, data types and column counts
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    1460 non-null   int64  
 18  OverallCond    1460

In [6]:
# SalePrice statistics
sale_price_count = dataset['SalePrice'].count()
print(f"How many SalePrice values do I have? Answer: {sale_price_count}")

sale_price_mean = dataset['SalePrice'].mean()
print(f"What is the average house price? Answer: {sale_price_mean:.3f}")

sale_price_median = dataset['SalePrice'].median()
print(f"What's the median house price? Answer: {sale_price_median}")

sale_price_std = dataset['SalePrice'].std()
print(f"How spread out are the prices? Answer: {sale_price_std:.3f}")

sale_price_min = dataset['SalePrice'].min()
print(f"What is the cheapest house? Answer: {sale_price_min}")

sale_price_max= dataset['SalePrice'].max()
print(f"What is the most expensive house? Answer: {sale_price_max}")

sale_price_null = dataset['SalePrice'].isnull().sum()
print(f"Is there a null value in the SalePrice column? Answer: ", "No" if sale_price_null == 0 else "Yes")

How many SalePrice values do I have? Answer: 1460
What is the average house price? Answer: 180921.196
What's the median house price? Answer: 163000.0
How spread out are the prices? Answer: 79442.503
What is the cheapest house? Answer: 34900
What is the most expensive house? Answer: 755000
Is there a null value in the SalePrice column? Answer:  No


In [7]:
# View Correlation of SalePrice to other features in descending order
dataset.corr(numeric_only=True)['SalePrice'].sort_values(ascending=False).head(50)

SalePrice        1.000000
OverallQual      0.790982
GrLivArea        0.708624
GarageCars       0.640409
GarageArea       0.623431
TotalBsmtSF      0.613581
1stFlrSF         0.605852
FullBath         0.560664
TotRmsAbvGrd     0.533723
YearBuilt        0.522897
YearRemodAdd     0.507101
GarageYrBlt      0.486362
MasVnrArea       0.477493
Fireplaces       0.466929
BsmtFinSF1       0.386420
LotFrontage      0.351799
WoodDeckSF       0.324413
2ndFlrSF         0.319334
OpenPorchSF      0.315856
HalfBath         0.284108
LotArea          0.263843
BsmtFullBath     0.227122
BsmtUnfSF        0.214479
BedroomAbvGr     0.168213
ScreenPorch      0.111447
PoolArea         0.092404
MoSold           0.046432
3SsnPorch        0.044584
BsmtFinSF2      -0.011378
BsmtHalfBath    -0.016844
MiscVal         -0.021190
Id              -0.021917
LowQualFinSF    -0.025606
YrSold          -0.028923
OverallCond     -0.077856
MSSubClass      -0.084284
EnclosedPorch   -0.128578
KitchenAbvGr    -0.135907
Name: SalePr

In [8]:
# Categorical Features
neighborhood_uniqe_values = dataset["Neighborhood"].nunique()
print(f"How many unique value are in Neighborhood? Answer: {neighborhood_uniqe_values}")

How many unique value are in Neighborhood? Answer: 25


In [9]:
# Grouping neighborhood and there average prices
dataset.groupby(["Neighborhood"])['SalePrice'].mean().sort_values(ascending=False)

Neighborhood
NoRidge    335295.317073
NridgHt    316270.623377
StoneBr    310499.000000
Timber     242247.447368
Veenker    238772.727273
Somerst    225379.837209
ClearCr    212565.428571
Crawfor    210624.725490
CollgCr    197965.773333
Blmngtn    194870.882353
Gilbert    192854.506329
NWAmes     189050.068493
SawyerW    186555.796610
Mitchel    156270.122449
NAmes      145847.080000
NPkVill    142694.444444
SWISU      142591.360000
Blueste    137500.000000
Sawyer     136793.135135
OldTown    128225.300885
Edwards    128219.700000
BrkSide    124834.051724
BrDale     104493.750000
IDOTRR     100123.783784
MeadowV     98576.470588
Name: SalePrice, dtype: float64

In [10]:
# Getting categorical feature unique values
dataset.select_dtypes('str').nunique().sort_values()

Street            2
Alley             2
CentralAir        2
Utilities         2
MasVnrType        3
LandSlope         3
PoolQC            3
PavedDrive        3
GarageFinish      3
BsmtQual          4
ExterQual         4
KitchenQual       4
BsmtCond          4
BsmtExposure      4
Fence             4
MiscFeature       4
LandContour       4
LotShape          4
FireplaceQu       5
Electrical        5
HeatingQC         5
GarageQual        5
GarageCond        5
MSZoning          5
LotConfig         5
BldgType          5
ExterCond         5
BsmtFinType1      6
RoofStyle         6
GarageType        6
Foundation        6
Heating           6
BsmtFinType2      6
SaleCondition     6
Functional        7
RoofMatl          8
HouseStyle        8
Condition2        8
SaleType          9
Condition1        9
Exterior1st      15
Exterior2nd      16
Neighborhood     25
dtype: int64

In [11]:
dataset.select_dtypes('str').isnull().sum().sort_values(ascending=False)

PoolQC           1453
MiscFeature      1406
Alley            1369
Fence            1179
MasVnrType        872
FireplaceQu       690
GarageType         81
GarageCond         81
GarageQual         81
GarageFinish       81
BsmtFinType2       38
BsmtExposure       38
BsmtFinType1       37
BsmtQual           37
BsmtCond           37
Electrical          1
KitchenQual         0
CentralAir          0
Functional          0
HeatingQC           0
Heating             0
PavedDrive          0
SaleType            0
MSZoning            0
Street              0
Condition2          0
LotShape            0
LandContour         0
Utilities           0
LotConfig           0
LandSlope           0
Neighborhood        0
Condition1          0
BldgType            0
Foundation          0
HouseStyle          0
RoofStyle           0
RoofMatl            0
Exterior1st         0
Exterior2nd         0
ExterQual           0
ExterCond           0
SaleCondition       0
dtype: int64

In [12]:
dataset['PoolQC'].value_counts()

PoolQC
Gd    3
Ex    2
Fa    2
Name: count, dtype: int64

In [13]:
dataset[dataset['PoolQC'].isnull()]['PoolArea'].value_counts()

PoolArea
0    1453
Name: count, dtype: int64

In [14]:
dataset['Alley'].value_counts()

Alley
Grvl    50
Pave    41
Name: count, dtype: int64

In [15]:
dataset[['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'GarageType', 'GarageCond', 'GarageQual', 'GarageFinish', 'BsmtFinType2', 'BsmtExposure', 'BsmtFinType1', 'BsmtQual', 'BsmtCond']].head(20)


,PoolQC,MiscFeature,Alley,Fence,FireplaceQu,GarageType,GarageCond,GarageQual,GarageFinish,BsmtFinType2,BsmtExposure,BsmtFinType1,BsmtQual,BsmtCond
0,NaN,NaN,NaN,NaN,NaN,Attchd,TA,TA,RFn,Unf,No,GLQ,Gd,TA
1,NaN,NaN,NaN,NaN,TA,Attchd,TA,TA,RFn,Unf,Gd,ALQ,Gd,TA
2,NaN,NaN,NaN,NaN,TA,Attchd,TA,TA,RFn,Unf,Mn,GLQ,Gd,TA
3,NaN,NaN,NaN,NaN,Gd,Detchd,TA,TA,Unf,Unf,No,ALQ,TA,Gd
4,NaN,NaN,NaN,NaN,TA,Attchd,TA,TA,RFn,Unf,Av,GLQ,Gd,TA
5,NaN,Shed,NaN,MnPrv,NaN,Attchd,TA,TA,Unf,Unf,No,GLQ,Gd,TA
6,NaN,NaN,NaN,NaN,Gd,Attchd,TA,TA,RFn,Unf,Av,GLQ,Ex,TA
7,NaN,Shed,NaN,NaN,TA,Attchd,TA,TA,RFn,BLQ,Mn,ALQ,Gd,TA
8,NaN,NaN,NaN,NaN,TA,Detchd,TA,Fa,Unf,Unf,No,Unf,TA,TA
9,NaN,NaN,NaN,NaN,TA,Attchd,TA,Gd,RFn,Unf,No,GLQ,TA,TA


In [16]:
dataset['MasVnrType'].value_counts(dropna=False)

MasVnrType
NaN        872
BrkFace    445
Stone      128
BrkCmn      15
Name: count, dtype: int64

In [17]:
dataset[dataset['MasVnrType'].isnull()]['MasVnrArea'].value_counts(dropna=False)

MasVnrArea
0.0      859
NaN        8
1.0        2
288.0      1
344.0      1
312.0      1
Name: count, dtype: int64

In [18]:
dataset[dataset['MasVnrType'].isnull() & (dataset['MasVnrArea'] != 0)][['MasVnrType', 'MasVnrArea']]

,MasVnrType,MasVnrArea
234,NaN,NaN
529,NaN,NaN
624,NaN,288.0
650,NaN,NaN
773,NaN,1.0
936,NaN,NaN
973,NaN,NaN
977,NaN,NaN
1230,NaN,1.0
1243,NaN,NaN


In [19]:
dataset.loc[
    [624, 773, 1230, 1300, 1334],
    ['MasVnrType', 'MasVnrArea', 'OverallQual',
     'YearBuilt', 'YearRemodAdd', 'Neighborhood', 
     'HouseStyle']]

,MasVnrType,MasVnrArea,OverallQual,YearBuilt,YearRemodAdd,Neighborhood,HouseStyle
624,NaN,288.0,6,1972,1972,NWAmes,2Story
773,NaN,1.0,5,1958,1958,NAmes,1Story
1230,NaN,1.0,5,1977,1977,Sawyer,1.5Fin
1300,NaN,344.0,7,1999,1999,Gilbert,2Story
1334,NaN,312.0,5,1970,1970,BrDale,2Story


In [20]:
# Investigating cell 624
dataset[(dataset['Neighborhood'] == 'NWAmes') &
(dataset['HouseStyle'] == '2Story')
][['MasVnrType', 'MasVnrArea', 'OverallQual', 'YearBuilt', 'YearRemodAdd']]

,MasVnrType,MasVnrArea,OverallQual,YearBuilt,YearRemodAdd
7,Stone,240.0,7,1973,1973
152,BrkFace,252.0,6,1971,1971
222,NaN,0.0,6,1975,1975
293,BrkFace,34.0,7,1977,1994
298,BrkFace,365.0,6,1968,1968
316,BrkFace,278.0,7,1980,1980
364,BrkFace,120.0,6,1976,1976
471,NaN,0.0,7,1977,1977
621,NaN,0.0,6,1974,1997
624,NaN,288.0,6,1972,1972


In [21]:
dataset[(dataset['Neighborhood'] == 'NWAmes') &
(dataset['MasVnrArea'].between(250, 350))
][['MasVnrType', 'MasVnrArea', 'Neighborhood', 'OverallQual', 'YearBuilt', 'YearRemodAdd']]

,MasVnrType,MasVnrArea,Neighborhood,OverallQual,YearBuilt,YearRemodAdd
135,BrkFace,288.0,NWAmes,7,1970,1970
152,BrkFace,252.0,NWAmes,6,1971,1971
316,BrkFace,278.0,NWAmes,7,1980,1980
455,BrkFace,320.0,NWAmes,7,1973,1973
516,BrkFace,333.0,NWAmes,6,1972,1972
624,NaN,288.0,NWAmes,6,1972,1972
643,BrkFace,263.0,NWAmes,5,1969,1969
745,BrkFace,289.0,NWAmes,8,1976,1996
927,BrkFace,342.0,NWAmes,7,1968,1968
988,BrkFace,298.0,NWAmes,6,1976,1976


In [22]:
dataset.loc[624, ['MasVnrType']] = 'BrkFace'

In [23]:
dataset.loc[624, ['MasVnrType', 'MasVnrArea']]

MasVnrType    BrkFace
MasVnrArea      288.0
Name: 624, dtype: object

In [24]:
# Investigating cell 773
dataset[(dataset['Neighborhood'] == 'NAmes') &
(dataset['HouseStyle'] == '1Story')&
(dataset['MasVnrArea'] > 0)
][['MasVnrType', 'MasVnrArea', 'OverallQual', 'YearBuilt', 'YearRemodAdd']]

,MasVnrType,MasVnrArea,OverallQual,YearBuilt,YearRemodAdd
14,BrkFace,212.0,6,1960,1960
16,BrkFace,180.0,6,1970,1970
37,BrkFace,650.0,5,1954,1990
40,BrkFace,101.0,6,1965,1965
55,BrkFace,272.0,6,1964,1964
66,BrkFace,287.0,7,1970,1989
70,BrkFace,1115.0,7,1973,1973
73,BrkFace,104.0,5,1954,2003
83,BrkCmn,66.0,5,1960,1960
91,BrkCmn,203.0,5,1961,1961


In [25]:
# Investigating cell 773
dataset[(dataset['Neighborhood'] == 'NAmes') &
(dataset['YearBuilt'].between(1955, 1961)) &
(dataset['MasVnrArea'] > 0)
][['MasVnrType', 'MasVnrArea', 'OverallQual', 'YearBuilt', 'YearRemodAdd', 'HouseStyle']]

,MasVnrType,MasVnrArea,OverallQual,YearBuilt,YearRemodAdd,HouseStyle
14,BrkFace,212.0,6,1960,1960,1Story
83,BrkCmn,66.0,5,1960,1960,1Story
91,BrkCmn,203.0,5,1961,1961,1Story
122,BrkFace,135.0,6,1958,1958,1Story
129,BrkFace,85.0,5,1958,1991,1Story
171,Stone,112.0,6,1960,1960,1Story
173,BrkCmn,491.0,6,1961,1961,1Story
215,BrkFace,64.0,5,1957,1996,1Story
230,BrkFace,220.0,6,1959,1959,1Story
273,BrkCmn,183.0,6,1958,1988,1Story


In [26]:
dataset[(dataset['MasVnrArea'] <= 10) &
(dataset['MasVnrArea'] > 0)][['MasVnrType', 'MasVnrArea', 'OverallQual', 'YearBuilt', 'YearRemodAdd', 'HouseStyle']]

,MasVnrType,MasVnrArea,OverallQual,YearBuilt,YearRemodAdd,HouseStyle
773,NaN,1.0,5,1958,1958,1Story
1230,NaN,1.0,5,1977,1977,1.5Fin


In [27]:
# Investigating cell 1230
dataset.loc[1230, ['Neighborhood', 'MasVnrType', 'MasVnrArea', 'OverallQual', 'YearBuilt', 'YearRemodAdd', 'HouseStyle', 'Exterior1st', 'Exterior2nd']]

Neighborhood     Sawyer
MasVnrType          NaN
MasVnrArea          1.0
OverallQual           5
YearBuilt          1977
YearRemodAdd       1977
HouseStyle       1.5Fin
Exterior1st     Plywood
Exterior2nd     Plywood
Name: 1230, dtype: object

In [28]:
dataset[
    (dataset['MasVnrType'].isnull()) &
    (dataset['MasVnrArea'] > 0)
    ][['Neighborhood', 'MasVnrType', 'MasVnrArea', 'OverallQual', 'YearBuilt', 'YearRemodAdd', 'HouseStyle', 'Exterior1st', 'Exterior2nd']]

,Neighborhood,MasVnrType,MasVnrArea,OverallQual,YearBuilt,YearRemodAdd,HouseStyle,Exterior1st,Exterior2nd
773,NAmes,NaN,1.0,5,1958,1958,1Story,Wd Sdng,Wd Sdng
1230,Sawyer,NaN,1.0,5,1977,1977,1.5Fin,Plywood,Plywood
1300,Gilbert,NaN,344.0,7,1999,1999,2Story,VinylSd,VinylSd
1334,BrDale,NaN,312.0,5,1970,1970,2Story,HdBoard,HdBoard


In [29]:
# Investigating row 1300
dataset[(dataset['Neighborhood'] == 'Gilbert') &
(dataset['YearBuilt'].between(1997, 2001))&
(dataset['MasVnrArea'] > 0)
][['MasVnrType', 'MasVnrArea', 'OverallQual', 'YearBuilt', 'YearRemodAdd', 'HouseStyle', 'MSSubClass', 'MSZoning', 'Street', 'Exterior1st', 'Exterior2nd']]

,MasVnrType,MasVnrArea,OverallQual,YearBuilt,YearRemodAdd,HouseStyle,MSSubClass,MSZoning,Street,Exterior1st,Exterior2nd
72,BrkFace,40.0,7,1998,1998,2Story,60,RL,Pave,VinylSd,VinylSd
131,BrkFace,40.0,6,2000,2000,2Story,60,RL,Pave,VinylSd,VinylSd
147,BrkFace,180.0,7,2001,2001,2Story,60,RL,Pave,VinylSd,VinylSd
255,BrkFace,302.0,7,1999,1999,2Story,60,RL,Pave,VinylSd,VinylSd
379,BrkFace,16.0,6,2000,2000,2Story,60,RL,Pave,VinylSd,VinylSd
665,BrkFace,40.0,8,2000,2000,2Story,60,RL,Pave,VinylSd,VinylSd
929,BrkFace,285.0,7,1997,1997,2Story,60,RL,Pave,HdBoard,HdBoard
941,BrkFace,298.0,7,1999,1999,2Story,60,RL,Pave,VinylSd,VinylSd
1300,NaN,344.0,7,1999,1999,2Story,60,RL,Pave,VinylSd,VinylSd
1433,BrkFace,318.0,6,2000,2000,2Story,60,RL,Pave,VinylSd,VinylSd


In [30]:
dataset.loc[1300, ['MasVnrType']] = 'BrkFace'

In [31]:
dataset.loc[1300, ['MasVnrType', 'MasVnrArea']]

MasVnrType    BrkFace
MasVnrArea      344.0
Name: 1300, dtype: object

In [32]:
# Investigating row 1334
dataset[(dataset['Neighborhood'] == 'BrDale') &
(dataset['YearBuilt'].between(1965, 1975))&
(dataset['MasVnrArea'] > 0)
][['MasVnrType', 'MasVnrArea', 'OverallQual', 'YearBuilt', 'YearRemodAdd', 'HouseStyle', 'MSSubClass', 'MSZoning', 'Street', 'Exterior1st', 'Exterior2nd']]

,MasVnrType,MasVnrArea,OverallQual,YearBuilt,YearRemodAdd,HouseStyle,MSSubClass,MSZoning,Street,Exterior1st,Exterior2nd
225,BrkFace,142.0,5,1971,1971,2Story,160,RM,Pave,HdBoard,HdBoard
227,BrkFace,127.0,6,1970,1970,2Story,160,RM,Pave,HdBoard,HdBoard
232,BrkFace,297.0,6,1972,1972,2Story,160,RM,Pave,HdBoard,HdBoard
235,BrkFace,604.0,6,1971,1971,2Story,160,RM,Pave,HdBoard,HdBoard
363,BrkFace,510.0,6,1972,2007,2Story,160,RM,Pave,HdBoard,HdBoard
430,BrkFace,232.0,6,1971,1971,2Story,160,RM,Pave,HdBoard,HdBoard
432,BrkFace,376.0,5,1971,1971,2Story,160,RM,Pave,HdBoard,HdBoard
500,BrkFace,285.0,6,1973,1973,2Story,160,RM,Pave,HdBoard,HdBoard
655,BrkFace,381.0,6,1971,1971,2Story,160,RM,Pave,HdBoard,ImStucc
837,BrkFace,158.0,6,1973,1973,2Story,160,RM,Pave,HdBoard,HdBoard


In [33]:
dataset.loc[1334, ['MasVnrType']] = 'BrkFace'

In [34]:
dataset.loc[1334, ['MasVnrType']]

MasVnrType    BrkFace
Name: 1334, dtype: str

In [35]:
# Investigating cell 773
dataset[(dataset['Neighborhood'] == 'NAmes') &
(dataset['HouseStyle'] == '1Story')&
(dataset['YearBuilt'].between(1956, 1960))&
(dataset['OverallQual'] == 5)&
(dataset['MasVnrArea'] > 0)
][['Neighborhood', 'MasVnrType', 'MasVnrArea', 'OverallQual', 'YearBuilt', 'YearRemodAdd', 'HouseStyle', 'MSSubClass', 'MSZoning', 'Street', 'Exterior1st', 'Exterior2nd']]

,Neighborhood,MasVnrType,MasVnrArea,OverallQual,YearBuilt,YearRemodAdd,HouseStyle,MSSubClass,MSZoning,Street,Exterior1st,Exterior2nd
83,NAmes,BrkCmn,66.0,5,1960,1960,1Story,20,RL,Pave,MetalSd,MetalSd
129,NAmes,BrkFace,85.0,5,1958,1991,1Story,20,RL,Pave,Plywood,Plywood
215,NAmes,BrkFace,64.0,5,1957,1996,1Story,20,RL,Pave,HdBoard,HdBoard
509,NAmes,BrkFace,132.0,5,1959,1959,1Story,20,RL,Pave,MetalSd,MetalSd
656,NAmes,BrkFace,54.0,5,1959,2006,1Story,20,RL,Pave,HdBoard,HdBoard
728,NAmes,BrkFace,95.0,5,1958,1958,1Story,90,RL,Pave,VinylSd,VinylSd
773,NAmes,NaN,1.0,5,1958,1958,1Story,20,RL,Pave,Wd Sdng,Wd Sdng
886,NAmes,BrkFace,122.0,5,1959,2005,1Story,90,RL,Pave,MetalSd,MetalSd
1026,NAmes,BrkFace,324.0,5,1960,1960,1Story,20,RL,Pave,MetalSd,HdBoard
1070,NAmes,BrkFace,120.0,5,1956,1956,1Story,20,RL,Pave,MetalSd,MetalSd


In [36]:
dataset.loc[773, ['MasVnrType']] = 'BrkFace'

In [37]:
dataset.loc[773, ['MasVnrType']]

MasVnrType    BrkFace
Name: 773, dtype: str

In [38]:
# Investigating cell 1230
dataset[(dataset['Neighborhood'] == 'Sawyer') &
(dataset['YearBuilt'].between(1970, 1985))&
(dataset['OverallQual'] == 5)&
(dataset['MasVnrArea'] > 0)
][['Neighborhood', 'MasVnrType', 'MasVnrArea', 'OverallQual', 'YearBuilt', 'YearRemodAdd', 'HouseStyle', 'MSSubClass', 'MSZoning', 'Street', 'Exterior1st', 'Exterior2nd']]

,Neighborhood,MasVnrType,MasVnrArea,OverallQual,YearBuilt,YearRemodAdd,HouseStyle,MSSubClass,MSZoning,Street,Exterior1st,Exterior2nd
622,Sawyer,BrkFace,153.0,5,1977,1977,1Story,20,RL,Pave,Plywood,Plywood
1230,Sawyer,NaN,1.0,5,1977,1977,1.5Fin,90,RL,Pave,Plywood,Plywood


In [39]:
# Investigating cell 1230
dataset[(dataset['Neighborhood'] == 'Sawyer') &
(dataset['Exterior1st'] == 'Plywood') &
(dataset['Exterior2nd'] == 'Plywood') &
(dataset['MasVnrArea'] > 0)
][['Neighborhood', 'MasVnrType', 'MasVnrArea', 'OverallQual', 'YearBuilt', 'YearRemodAdd', 'HouseStyle', 'MSSubClass', 'MSZoning', 'Street']]

,Neighborhood,MasVnrType,MasVnrArea,OverallQual,YearBuilt,YearRemodAdd,HouseStyle,MSSubClass,MSZoning,Street
538,Sawyer,BrkFace,188.0,5,1968,1968,1Story,20,RL,Pave
622,Sawyer,BrkFace,153.0,5,1977,1977,1Story,20,RL,Pave
796,Sawyer,BrkFace,148.0,6,1977,1977,1Story,20,RL,Pave
1230,Sawyer,NaN,1.0,5,1977,1977,1.5Fin,90,RL,Pave


In [40]:
dataset.loc[1230, ['MasVnrType']] = 'BrkFace'

In [41]:
dataset.loc[1230, ['MasVnrType']]

MasVnrType    BrkFace
Name: 1230, dtype: str

In [42]:
dataset.loc[[624, 773, 1230, 1300, 1334], ['Neighborhood','MasVnrType']]

,Neighborhood,MasVnrType
624,NWAmes,BrkFace
773,NAmes,BrkFace
1230,Sawyer,BrkFace
1300,Gilbert,BrkFace
1334,BrDale,BrkFace


In [43]:
dataset[dataset['MasVnrType'].isnull() &
dataset['MasVnrArea'].isnull()][['Neighborhood', 'OverallQual', 'YearBuilt', 'YearRemodAdd', 'HouseStyle', 'MSSubClass', 'MSZoning', 'Street', 'Exterior1st', 'Exterior2nd']]


,Neighborhood,OverallQual,YearBuilt,YearRemodAdd,HouseStyle,MSSubClass,MSZoning,Street,Exterior1st,Exterior2nd
234,Gilbert,6,2002,2002,2Story,60,RL,Pave,VinylSd,VinylSd
529,Crawfor,6,1957,1975,1Story,20,RL,Pave,Wd Sdng,Stone
650,Somerst,7,2007,2007,2Story,60,FV,Pave,CemntBd,CmentBd
936,SawyerW,7,2003,2003,1Story,20,RL,Pave,VinylSd,VinylSd
973,Somerst,7,2007,2008,1Story,20,FV,Pave,CemntBd,CmentBd
977,Somerst,7,2006,2007,1Story,120,FV,Pave,VinylSd,VinylSd
1243,NridgHt,10,2006,2006,1Story,20,RL,Pave,VinylSd,VinylSd
1278,CollgCr,8,2002,2002,2Story,60,RL,Pave,VinylSd,VinylSd


In [44]:
dataset.groupby('MasVnrType', dropna=False)['MasVnrArea'].agg(
    ['count', 'min', 'mean', 'max']
)

,count,min,mean,max
MasVnrType,,,,
BrkCmn,15,41.0,247.666667,621.0
BrkFace,450,0.0,258.233333,1600.0
Stone,128,0.0,239.304688,860.0
NaN,859,0.0,0.000000,0.0


In [45]:
dataset['MasVnrArea'].isnull().sum()

np.int64(8)

In [46]:
dataset[dataset['MasVnrArea'].isnull()]['MasVnrType'].value_counts(dropna=False)

MasVnrType
NaN    8
Name: count, dtype: int64

In [47]:
dataset[['MasVnrType', 'MasVnrArea']].isnull().sum()

MasVnrType    867
MasVnrArea      8
dtype: int64

In [48]:
dataset['MasVnrType'].value_counts(dropna=False)

MasVnrType
NaN        867
BrkFace    450
Stone      128
BrkCmn      15
Name: count, dtype: int64

In [49]:
dataset[dataset['MasVnrArea'] == 0]['MasVnrType'].value_counts(dropna=False)

MasVnrType
NaN        859
BrkFace      1
Stone        1
Name: count, dtype: int64

In [50]:
dataset[(dataset['MasVnrArea'] == 0) &
(dataset['MasVnrType'].notnull())][['Neighborhood', 'MasVnrType', 'MasVnrArea', 'OverallQual', 'YearBuilt', 'YearRemodAdd', 'HouseStyle', 'MSSubClass', 'MSZoning', 'Street', 'Exterior1st', 'Exterior2nd']]

,Neighborhood,MasVnrType,MasVnrArea,OverallQual,YearBuilt,YearRemodAdd,HouseStyle,MSSubClass,MSZoning,Street,Exterior1st,Exterior2nd
688,StoneBr,BrkFace,0.0,8,2007,2007,1Story,20,RL,Pave,MetalSd,MetalSd
1241,Somerst,Stone,0.0,7,2007,2007,1Story,20,RL,Pave,VinylSd,VinylSd


In [51]:
dataset[(dataset['Neighborhood'] == 'StoneBr') &
(dataset['YearBuilt'].between(2000, 2010))][['Neighborhood', 'MasVnrType', 'MasVnrArea', 'YearBuilt', 'OverallQual', 'HouseStyle', 'Exterior1st', 'Exterior2nd']]

,Neighborhood,MasVnrType,MasVnrArea,YearBuilt,OverallQual,HouseStyle,Exterior1st,Exterior2nd
58,StoneBr,BrkFace,1031.0,2006,10,2Story,VinylSd,VinylSd
178,StoneBr,Stone,748.0,2008,9,1Story,VinylSd,VinylSd
189,StoneBr,NaN,0.0,2001,8,1Story,CemntBd,CmentBd
336,StoneBr,Stone,200.0,2005,9,1Story,VinylSd,VinylSd
378,StoneBr,Stone,350.0,2010,9,1Story,VinylSd,VinylSd
474,StoneBr,NaN,0.0,2000,8,1Story,CemntBd,CmentBd
566,StoneBr,BrkFace,245.0,2005,9,2Story,VinylSd,VinylSd
595,StoneBr,BrkFace,238.0,2005,8,1Story,VinylSd,Other
678,StoneBr,Stone,464.0,2008,8,1Story,VinylSd,VinylSd
688,StoneBr,BrkFace,0.0,2007,8,1Story,MetalSd,MetalSd


In [52]:
dataset.groupby(['MasVnrType',	'MasVnrArea'], dropna=False).size().sort_values(ascending=False)

MasVnrType  MasVnrArea
NaN         0.0           859
            NaN             8
BrkFace     120.0           7
            180.0           7
            16.0            6
            108.0           6
            80.0            6
            360.0           5
            320.0           5
            170.0           5
            72.0            5
            40.0            4
            196.0           4
            456.0           4
            340.0           4
            168.0           4
            252.0           4
            132.0           3
            130.0           3
Stone       186.0           3
BrkFace     116.0           3
Stone       72.0            3
            200.0           3
BrkFace     106.0           3
            136.0           3
            99.0            3
            90.0            3
            85.0            3
            84.0            3
            104.0           3
            176.0           3
            148.0           3
            169.0

In [53]:
dataset[
    (dataset['MasVnrArea'] == 0) &
    (dataset['MasVnrType'].isnull())].shape[0]

859

In [54]:
dataset.loc[
    (dataset['MasVnrArea'] == 0) &
    (dataset['MasVnrType'].isnull()),
    'MasVnrType'
    ] = 'None'

In [55]:
dataset['MasVnrType'].value_counts(dropna=False)

MasVnrType
None       859
BrkFace    450
Stone      128
BrkCmn      15
NaN          8
Name: count, dtype: int64

In [56]:
dataset[dataset['MasVnrArea'].isnull()][['Neighborhood', 'MasVnrType', 'MasVnrArea', 'YearBuilt', 'OverallQual', 'HouseStyle', 'MSSubClass', 'Exterior1st', 'Exterior2nd']]

,Neighborhood,MasVnrType,MasVnrArea,YearBuilt,OverallQual,HouseStyle,MSSubClass,Exterior1st,Exterior2nd
234,Gilbert,NaN,NaN,2002,6,2Story,60,VinylSd,VinylSd
529,Crawfor,NaN,NaN,1957,6,1Story,20,Wd Sdng,Stone
650,Somerst,NaN,NaN,2007,7,2Story,60,CemntBd,CmentBd
936,SawyerW,NaN,NaN,2003,7,1Story,20,VinylSd,VinylSd
973,Somerst,NaN,NaN,2007,7,1Story,20,CemntBd,CmentBd
977,Somerst,NaN,NaN,2006,7,1Story,120,VinylSd,VinylSd
1243,NridgHt,NaN,NaN,2006,10,1Story,20,VinylSd,VinylSd
1278,CollgCr,NaN,NaN,2002,8,2Story,60,VinylSd,VinylSd


In [57]:
# Investigating row 234
dataset[(dataset['Neighborhood'] == 'Gilbert') &
(dataset['HouseStyle'] == '2Story') &
(dataset['MSSubClass'] == 60) &
(dataset['Exterior1st'] == 'VinylSd') &
(dataset['YearBuilt'] == 2002) &
(dataset['OverallQual'] == 6) &
(dataset['Exterior2nd'] == 'VinylSd')][['MasVnrType', 'MasVnrArea', 'YearBuilt', 'OverallQual', 'HouseStyle', 'MSSubClass','Exterior1st', 'Exterior2nd']]

,MasVnrType,MasVnrArea,YearBuilt,OverallQual,HouseStyle,MSSubClass,Exterior1st,Exterior2nd
221,None,0.0,2002,6,2Story,60,VinylSd,VinylSd
234,NaN,NaN,2002,6,2Story,60,VinylSd,VinylSd
616,None,0.0,2002,6,2Story,60,VinylSd,VinylSd


In [58]:
dataset.loc[234, ['MasVnrType',	'MasVnrArea']] = ['BrkFace', 0]

In [59]:
dataset.loc[234, ['MasVnrType',	'MasVnrArea']]

MasVnrType    BrkFace
MasVnrArea        0.0
Name: 234, dtype: object

In [60]:
# Investigating row 529
dataset[(dataset['Neighborhood'] == 'Crawfor') &
(dataset['HouseStyle'] == '1Story') &
(dataset['MSSubClass'] == 20) &
(dataset['Exterior1st'] == 'Wd Sdng') &
(dataset['Exterior2nd'] == 'Stone')][['MasVnrType', 'MasVnrArea', 'YearBuilt', 'OverallQual', 'HouseStyle', 'MSSubClass','Exterior1st', 'Exterior2nd']]

,MasVnrType,MasVnrArea,YearBuilt,OverallQual,HouseStyle,MSSubClass,Exterior1st,Exterior2nd
529,NaN,NaN,1957,6,1Story,20,Wd Sdng,Stone


In [61]:
# Investigating row 529
dataset[(dataset['Neighborhood'] == 'Crawfor') &
(dataset['Exterior2nd'] == 'Stone')][['MasVnrType', 'MasVnrArea', 'YearBuilt', 'OverallQual', 'HouseStyle', 'MSSubClass','Exterior1st', 'Exterior2nd']]

,MasVnrType,MasVnrArea,YearBuilt,OverallQual,HouseStyle,MSSubClass,Exterior1st,Exterior2nd
529,NaN,NaN,1957,6,1Story,20,Wd Sdng,Stone
1152,None,0.0,1956,6,1Story,20,Stone,Stone
1343,None,0.0,1928,6,1.5Fin,50,BrkFace,Stone


In [62]:
dataset.loc[529, ['MasVnrType', 'MasVnrArea']] = ['None', 0]

In [63]:
dataset.loc[529, ['MasVnrType', 'MasVnrArea', 'YearBuilt', 'OverallQual', 'HouseStyle', 'MSSubClass','Exterior1st', 'Exterior2nd']]

MasVnrType        None
MasVnrArea         0.0
YearBuilt         1957
OverallQual          6
HouseStyle      1Story
MSSubClass          20
Exterior1st    Wd Sdng
Exterior2nd      Stone
Name: 529, dtype: object

In [64]:
# Investigating row 650
dataset[(dataset['Neighborhood'] == 'Somerst') &
(dataset['HouseStyle'] == '2Story') &
(dataset['MSSubClass'] == 60) &
(dataset['Exterior1st'] == 'CemntBd') &
(dataset['Exterior2nd'] == 'CmentBd')][['MasVnrType', 'MasVnrArea', 'YearBuilt', 'OverallQual', 'HouseStyle', 'MSSubClass','Exterior1st', 'Exterior2nd']]

,MasVnrType,MasVnrArea,YearBuilt,OverallQual,HouseStyle,MSSubClass,Exterior1st,Exterior2nd
377,None,0.0,2004,8,2Story,60,CemntBd,CmentBd
399,Stone,100.0,2006,7,2Story,60,CemntBd,CmentBd
650,NaN,NaN,2007,7,2Story,60,CemntBd,CmentBd
875,None,0.0,2007,8,2Story,60,CemntBd,CmentBd
1374,None,0.0,2005,7,2Story,60,CemntBd,CmentBd


In [65]:
dataset.loc[650, ['MasVnrType',	'MasVnrArea']] = ['None', 0]

In [66]:
dataset.loc[650, ['MasVnrType',	'MasVnrArea']]

MasVnrType    None
MasVnrArea     0.0
Name: 650, dtype: object

In [67]:
# Investigating row 936
dataset[(dataset['Neighborhood'] == 'SawyerW') &
(dataset['HouseStyle'] == '1Story') &
(dataset['MSSubClass'] == 20) &
(dataset['Exterior1st'] == 'VinylSd') &
(dataset['Exterior2nd'] == 'VinylSd')][['MasVnrType', 'MasVnrArea', 'YearBuilt', 'OverallQual', 'HouseStyle', 'MSSubClass','Exterior1st', 'Exterior2nd']]

,MasVnrType,MasVnrArea,YearBuilt,OverallQual,HouseStyle,MSSubClass,Exterior1st,Exterior2nd
18,None,0.0,2004,5,1Story,20,VinylSd,VinylSd
60,None,0.0,2004,6,1Story,20,VinylSd,VinylSd
148,BrkFace,120.0,2004,7,1Story,20,VinylSd,VinylSd
742,BrkFace,108.0,2000,7,1Story,20,VinylSd,VinylSd
827,None,0.0,2001,7,1Story,20,VinylSd,VinylSd
936,NaN,NaN,2003,7,1Story,20,VinylSd,VinylSd
951,None,0.0,1965,5,1Story,20,VinylSd,VinylSd
1238,None,0.0,2005,6,1Story,20,VinylSd,VinylSd
1341,None,0.0,2003,6,1Story,20,VinylSd,VinylSd
1348,None,0.0,1998,7,1Story,20,VinylSd,VinylSd


In [68]:
dataset.loc[936, ['MasVnrType',	'MasVnrArea']] = ['None', 0]

In [69]:
dataset.loc[936, ['MasVnrType',	'MasVnrArea']]

MasVnrType    None
MasVnrArea     0.0
Name: 936, dtype: object

In [70]:

# Investigating row 973
dataset[(dataset['Neighborhood'] == 'Somerst') &
(dataset['HouseStyle'] == '1Story') &
(dataset['MSSubClass'] == 20) &
(dataset['Exterior1st'] == 'CemntBd') &
(dataset['Exterior2nd'] == 'CmentBd')][['MasVnrType', 'MasVnrArea', 'YearBuilt', 'OverallQual', 'HouseStyle', 'MSSubClass','Exterior1st', 'Exterior2nd']]

,MasVnrType,MasVnrArea,YearBuilt,OverallQual,HouseStyle,MSSubClass,Exterior1st,Exterior2nd
196,Stone,205.0,2007,7,1Story,20,CemntBd,CmentBd
644,Stone,162.0,2009,9,1Story,20,CemntBd,CmentBd
793,Stone,140.0,2007,8,1Story,20,CemntBd,CmentBd
973,NaN,NaN,2007,7,1Story,20,CemntBd,CmentBd
1217,Stone,72.0,2009,8,1Story,20,CemntBd,CmentBd
1451,Stone,194.0,2008,8,1Story,20,CemntBd,CmentBd


In [71]:

# Investigating row 973
dataset[(dataset['Neighborhood'] == 'Somerst') &
(dataset['HouseStyle'] == '1Story') &
(dataset['MSSubClass'] == 20) &
(dataset['Exterior1st'] == 'CemntBd') &
(dataset['MasVnrType'] == 'Stone') &
(dataset['Exterior2nd'] == 'CmentBd')]['MasVnrArea'].agg(['count', 'mean', 'median', 'min', 'max'])

count       5.0
mean      154.6
median    162.0
min        72.0
max       205.0
Name: MasVnrArea, dtype: float64

In [72]:
dataset.loc[973, ['MasVnrArea']] = 162

In [73]:
dataset.loc[973, ['MasVnrType']] = 'Stone'

In [74]:
dataset.loc[973, ['MasVnrType', 'MasVnrArea']]

MasVnrType    Stone
MasVnrArea    162.0
Name: 973, dtype: object

In [75]:
# Investigating row 977
dataset[(dataset['Neighborhood'] == 'Somerst') &
(dataset['HouseStyle'] == '1Story') &
(dataset['MSSubClass'] == 120) &
(dataset['Exterior1st'] == 'VinylSd') &
(dataset['Exterior2nd'] == 'VinylSd')][['MasVnrType', 'MasVnrArea', 'YearBuilt', 'OverallQual', 'HouseStyle', 'MSSubClass','Exterior1st', 'Exterior2nd']]

,MasVnrType,MasVnrArea,YearBuilt,OverallQual,HouseStyle,MSSubClass,Exterior1st,Exterior2nd
977,NaN,NaN,2006,7,1Story,120,VinylSd,VinylSd
1317,None,0.0,2006,7,1Story,120,VinylSd,VinylSd


In [76]:
dataset.loc[977, ['MasVnrType',	'MasVnrArea']] = ['None', 0]

In [77]:
dataset.loc[977, ['MasVnrType',	'MasVnrArea']]

MasVnrType    None
MasVnrArea     0.0
Name: 977, dtype: object

In [78]:
# Investigating row 1243
dataset[(dataset['Neighborhood'] == 'NridgHt') &
(dataset['Exterior1st'] == 'VinylSd') &
(dataset['YearBuilt'] == 2006) &
(dataset['Exterior2nd'] == 'VinylSd')][['MasVnrType', 'MasVnrArea', 'OverallQual']]

,MasVnrType,MasVnrArea,OverallQual
62,Stone,178.0,8
320,Stone,370.0,9
408,BrkFace,350.0,7
468,Stone,248.0,8
473,BrkFace,304.0,8
477,BrkFace,772.0,9
552,Stone,250.0,8
631,Stone,108.0,8
774,BrkFace,375.0,8
1243,NaN,NaN,10


In [79]:
# Investigating row 1243
dataset[(dataset['Neighborhood'] == 'NridgHt') &
(dataset['Exterior1st'] == 'VinylSd') &
(dataset['YearBuilt'] == 2006) &
(dataset['MasVnrType'] == 'Stone') &
(dataset['Exterior2nd'] == 'VinylSd')][['MasVnrArea']].agg(['count', 'mean', 'median', 'min', 'max'])

,MasVnrArea
count,7.000000
mean,320.285714
median,250.000000
min,108.000000
max,788.000000


In [80]:
dataset.loc[1243, ['MasVnrType', 'MasVnrArea']] = ['Stone', 320.29]

In [81]:
dataset.loc[1243, ['MasVnrType', 'MasVnrArea']]

MasVnrType     Stone
MasVnrArea    320.29
Name: 1243, dtype: object

In [82]:
# Investigating row 1278
dataset[(dataset['Neighborhood'] == 'CollgCr') &
(dataset['HouseStyle'] == '2Story') &
(dataset['MSSubClass'] == 60) &
(dataset['Exterior1st'] == 'VinylSd') &
(dataset['YearBuilt'] == 2002) &
(dataset['OverallQual'] == 8) &
(dataset['Exterior2nd'] == 'VinylSd')][['MasVnrType', 'MasVnrArea', 'YearBuilt', 'OverallQual', 'HouseStyle', 'MSSubClass','Exterior1st', 'Exterior2nd']]

,MasVnrType,MasVnrArea,YearBuilt,OverallQual,HouseStyle,MSSubClass,Exterior1st,Exterior2nd
1054,BrkFace,210.0,2002,8,2Story,60,VinylSd,VinylSd
1278,NaN,NaN,2002,8,2Story,60,VinylSd,VinylSd
1342,BrkFace,149.0,2002,8,2Story,60,VinylSd,VinylSd


In [83]:
# Investigating row 1278
dataset[(dataset['Neighborhood'] == 'CollgCr') &
(dataset['HouseStyle'] == '2Story') &
(dataset['MSSubClass'] == 60) &
(dataset['Exterior1st'] == 'VinylSd') &
(dataset['YearBuilt'] == 2002) &
(dataset['OverallQual'] == 8) &
(dataset['Exterior2nd'] == 'VinylSd')][['MasVnrArea']].agg(['count', 'mean', 'median', 'min', 'max'])

,MasVnrArea
count,2.0
mean,179.5
median,179.5
min,149.0
max,210.0


In [84]:
dataset.loc[1278, ['MasVnrType', 'MasVnrArea']] = ['BrkFace', 179.5]

In [85]:
dataset.loc[1278, ['MasVnrType', 'MasVnrArea']]

MasVnrType    BrkFace
MasVnrArea      179.5
Name: 1278, dtype: object

In [86]:
dataset.loc[1278, ['OverallQual', 'GarageArea', 'GrLivArea', 'TotalBsmtSF','1stFlrSF', 'SalePrice']]

OverallQual         8
GarageArea        577
GrLivArea        2031
TotalBsmtSF      1128
1stFlrSF         1128
SalePrice      237000
Name: 1278, dtype: int64

In [87]:
dataset.loc[[234, 529, 650, 936, 973, 977, 1243, 1278], ['MasVnrType', 'MasVnrArea']]

,MasVnrType,MasVnrArea
234,BrkFace,0.00
529,None,0.00
650,None,0.00
936,None,0.00
973,Stone,162.00
977,None,0.00
1243,Stone,320.29
1278,BrkFace,179.50


In [88]:
dataset.isnull().sum().sort_values(ascending=False)

PoolQC           1453
MiscFeature      1406
Alley            1369
Fence            1179
FireplaceQu       690
LotFrontage       259
GarageYrBlt        81
GarageType         81
GarageFinish       81
GarageQual         81
GarageCond         81
BsmtFinType2       38
BsmtExposure       38
BsmtCond           37
BsmtQual           37
BsmtFinType1       37
Electrical          1
TotRmsAbvGrd        0
Id                  0
KitchenQual         0
Functional          0
BedroomAbvGr        0
HalfBath            0
FullBath            0
BsmtHalfBath        0
BsmtFullBath        0
KitchenAbvGr        0
GarageCars          0
Fireplaces          0
LowQualFinSF        0
GarageArea          0
PavedDrive          0
WoodDeckSF          0
OpenPorchSF         0
EnclosedPorch       0
3SsnPorch           0
ScreenPorch         0
PoolArea            0
MiscVal             0
MoSold              0
YrSold              0
SaleType            0
SaleCondition       0
GrLivArea           0
HeatingQC           0
2ndFlrSF  

In [89]:
dataset[dataset['MasVnrType'].isnull() |
dataset['MasVnrArea'].isnull()][['MasVnrType', 'MasVnrArea']]

,MasVnrType,MasVnrArea


In [90]:
dataset[dataset['PoolQC'].isnull()]['PoolArea'].value_counts()

PoolArea
0    1453
Name: count, dtype: int64

In [91]:
dataset[dataset['BsmtQual'].isnull()]['TotalBsmtSF'].value_counts()

TotalBsmtSF
0    37
Name: count, dtype: int64

In [92]:
dataset[dataset['GarageQual'].isnull()]['GarageArea'].value_counts()

GarageArea
0    81
Name: count, dtype: int64

In [93]:
dataset[dataset['FireplaceQu'].isnull()]['Fireplaces'].value_counts()

Fireplaces
0    690
Name: count, dtype: int64

In [94]:
dataset[dataset['MiscFeature'].isnull()]['MiscVal'].value_counts()

MiscVal
0    1406
Name: count, dtype: int64

In [95]:
dataset[dataset['Fence'].isnull()][['Fence', 'LotArea', 'OverallQual']].head(20)

,Fence,LotArea,OverallQual
0,NaN,8450,7
1,NaN,9600,6
2,NaN,11250,7
3,NaN,9550,7
4,NaN,14260,8
6,NaN,10084,8
7,NaN,10382,7
8,NaN,6120,7
9,NaN,7420,5
10,NaN,11200,5


In [96]:
dataset['Electrical'].isnull().sum()

np.int64(1)

In [97]:
missing = dataset.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtFinType2      38
BsmtExposure      38
BsmtFinType1      37
BsmtCond          37
BsmtQual          37
Electrical         1
dtype: int64

In [98]:
# Inspecting garage without values
dataset[dataset['GarageType'].isnull()][['GarageArea', 'GarageCars', 'GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageQual', 'GarageCond']].head(20)

,GarageArea,GarageCars,GarageType,GarageYrBlt,GarageFinish,GarageQual,GarageCond
39,0,0,NaN,NaN,NaN,NaN,NaN
48,0,0,NaN,NaN,NaN,NaN,NaN
78,0,0,NaN,NaN,NaN,NaN,NaN
88,0,0,NaN,NaN,NaN,NaN,NaN
89,0,0,NaN,NaN,NaN,NaN,NaN
99,0,0,NaN,NaN,NaN,NaN,NaN
108,0,0,NaN,NaN,NaN,NaN,NaN
125,0,0,NaN,NaN,NaN,NaN,NaN
127,0,0,NaN,NaN,NaN,NaN,NaN
140,0,0,NaN,NaN,NaN,NaN,NaN


In [99]:
# Inspecting basement without values
dataset[dataset['BsmtQual'].isnull()][['TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'BsmtQual', 'BsmtExposure', 'BsmtFinType2', 'BsmtCond', 'BsmtFinType1']].head(20)

,TotalBsmtSF,BsmtFullBath,BsmtHalfBath,BsmtQual,BsmtExposure,BsmtFinType2,BsmtCond,BsmtFinType1
17,0,0,0,NaN,NaN,NaN,NaN,NaN
39,0,0,0,NaN,NaN,NaN,NaN,NaN
90,0,0,0,NaN,NaN,NaN,NaN,NaN
102,0,0,0,NaN,NaN,NaN,NaN,NaN
156,0,0,0,NaN,NaN,NaN,NaN,NaN
182,0,0,0,NaN,NaN,NaN,NaN,NaN
259,0,0,0,NaN,NaN,NaN,NaN,NaN
342,0,0,0,NaN,NaN,NaN,NaN,NaN
362,0,0,0,NaN,NaN,NaN,NaN,NaN
371,0,0,0,NaN,NaN,NaN,NaN,NaN


In [100]:
# Inspecting fireplace without values
dataset[dataset['FireplaceQu'].isnull()][['FireplaceQu', 'Fireplaces']].head(20)

,FireplaceQu,Fireplaces
0,NaN,0
5,NaN,0
10,NaN,0
12,NaN,0
15,NaN,0
17,NaN,0
18,NaN,0
19,NaN,0
26,NaN,0
29,NaN,0


In [101]:
# Inspecting pool without values
dataset[dataset['PoolQC'].isnull()][['PoolQC', 'PoolArea']].head(20)

,PoolQC,PoolArea
0,NaN,0
1,NaN,0
2,NaN,0
3,NaN,0
4,NaN,0
5,NaN,0
6,NaN,0
7,NaN,0
8,NaN,0
9,NaN,0


In [102]:
# Inspecting misc without values
dataset[dataset['MiscFeature'].isnull()][['MiscFeature', 'MiscVal']].head(20)

,MiscFeature,MiscVal
0,NaN,0
1,NaN,0
2,NaN,0
3,NaN,0
4,NaN,0
6,NaN,0
8,NaN,0
9,NaN,0
10,NaN,0
11,NaN,0


In [103]:
# Inspecting Fence without values
dataset['Fence'].value_counts(dropna=False)

Fence
NaN      1179
MnPrv     157
GdPrv      59
GdWo       54
MnWw       11
Name: count, dtype: int64

In [104]:
# Inspecting Fence without values
dataset['Alley'].value_counts(dropna=False)

Alley
NaN     1369
Grvl      50
Pave      41
Name: count, dtype: int64

In [105]:
absent_features = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageCond', 'GarageQual', 'BsmtFinType2', 'BsmtExposure', 'BsmtCond', 'BsmtFinType1', 'BsmtQual']

In [106]:
missing[missing >0].sort_values(ascending=False)

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtFinType2      38
BsmtExposure      38
BsmtFinType1      37
BsmtCond          37
BsmtQual          37
Electrical         1
dtype: int64

In [107]:
# Investigating Pool
dataset.groupby('PoolQC', dropna=False)['SalePrice'].mean()

PoolQC
Ex     490000.000000
Fa     215500.000000
Gd     201990.000000
NaN    180404.663455
Name: SalePrice, dtype: float64

In [108]:
# Investigating Garage
dataset.groupby('GarageType', dropna=False)['SalePrice'].agg(['count', 'mean'])

,count,mean
GarageType,,
2Types,6,151283.333333
Attchd,870,202892.656322
Basment,19,160570.684211
BuiltIn,88,254751.738636
CarPort,9,109962.111111
Detchd,387,134091.162791
NaN,81,103317.283951


In [109]:
dataset[dataset['GarageType'].isnull()][['GarageArea', 'GarageCars', 'GarageYrBlt']].head(20)

,GarageArea,GarageCars,GarageYrBlt
39,0,0,NaN
48,0,0,NaN
78,0,0,NaN
88,0,0,NaN
89,0,0,NaN
99,0,0,NaN
108,0,0,NaN
125,0,0,NaN
127,0,0,NaN
140,0,0,NaN


In [110]:
absent_features = dataset.select_dtypes('str').isnull().sum().sort_values(ascending=False)
absent_features[absent_features > 0]

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
FireplaceQu      690
GarageType        81
GarageFinish      81
GarageCond        81
GarageQual        81
BsmtFinType2      38
BsmtExposure      38
BsmtCond          37
BsmtFinType1      37
BsmtQual          37
Electrical         1
dtype: int64

In [111]:
dataset[dataset['Electrical'].isnull()]

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
1379,1380,80,RL,73.0,9735,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2008,WD,Normal,167500


In [112]:
dataset['Electrical'].value_counts()

Electrical
SBrkr    1334
FuseA      94
FuseF      27
FuseP       3
Mix         1
Name: count, dtype: int64

In [113]:
dataset.loc[1379, ['Electrical', 'OverallQual', 'YearBuilt', 'YearRemodAdd',
                   'Neighborhood', 'HouseStyle', 'CentralAir']]

Electrical         NaN
OverallQual          5
YearBuilt         2006
YearRemodAdd      2007
Neighborhood    Timber
HouseStyle        SLvl
CentralAir           Y
Name: 1379, dtype: object

In [114]:
dataset[dataset['Neighborhood'] == 'Timber']['Electrical'].value_counts()

Electrical
SBrkr    36
FuseA     1
Name: count, dtype: int64

In [115]:
dataset[dataset['YearBuilt'] >= 2000]['Electrical'].value_counts()

Electrical
SBrkr    387
Name: count, dtype: int64

In [116]:
dataset[
    (dataset['Neighborhood'] == 'Timber') & 
    (dataset['YearBuilt'] >= 2000) & 
    (dataset['HouseStyle'] == 'SLvl')]['Electrical'].value_counts(dropna=False)

Electrical
SBrkr    1
NaN      1
Name: count, dtype: int64

In [117]:
dataset[dataset['Neighborhood'] == 'Timber'][['Electrical', 'SalePrice']].sort_values('SalePrice')

,Electrical,SalePrice
944,FuseA,137500
1396,SBrkr,160000
1379,NaN,167500
41,SBrkr,170000
429,SBrkr,175000
530,SBrkr,175000
695,SBrkr,176000
1281,SBrkr,180000
174,SBrkr,184000
1184,SBrkr,186700


In [118]:
dataset.loc[1379, ['Electrical']] = 'SBrkr'

In [119]:
dataset.loc[1379, ['Electrical']]

Electrical    SBrkr
Name: 1379, dtype: str

In [120]:
dataset['Electrical'].isnull().sum()

np.int64(0)

In [121]:
absent_features = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageCond', 'GarageQual', 'BsmtFinType2', 'BsmtExposure', 'BsmtCond', 'BsmtFinType1', 'BsmtQual']

In [122]:
missing_values_without_zeros = dataset.isnull().sum()
missing_values_without_zeros[missing_values_without_zeros > 0].sort_values(ascending=False)

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
dtype: int64

In [123]:
dataset[absent_features].dtypes

PoolQC              str
MiscFeature         str
Alley               str
Fence               str
FireplaceQu         str
GarageType          str
GarageYrBlt     float64
GarageFinish        str
GarageCond          str
GarageQual          str
BsmtFinType2        str
BsmtExposure        str
BsmtCond            str
BsmtFinType1        str
BsmtQual            str
dtype: object

In [124]:
dataset['BsmtQual'] = dataset['BsmtQual'].fillna('None')

In [125]:
dataset['BsmtQual'].isnull().value_counts(dropna=False)

BsmtQual
False    1460
Name: count, dtype: int64

In [126]:
dataset['BsmtQual'].value_counts()

BsmtQual
TA      649
Gd      618
Ex      121
None     37
Fa       35
Name: count, dtype: int64

In [127]:
dataset['BsmtFinType1'] = dataset['BsmtFinType1'].fillna('None')

In [128]:
dataset['BsmtFinType1'].isnull().value_counts(dropna=False)

BsmtFinType1
False    1460
Name: count, dtype: int64

In [129]:
dataset['BsmtFinType1'].value_counts()

BsmtFinType1
Unf     430
GLQ     418
ALQ     220
BLQ     148
Rec     133
LwQ      74
None     37
Name: count, dtype: int64

In [130]:
dataset['BsmtCond'] = dataset['BsmtCond'].fillna('None')

In [131]:
dataset['BsmtCond'].isnull().value_counts(dropna=False)

BsmtCond
False    1460
Name: count, dtype: int64

In [132]:
dataset['BsmtCond'].value_counts()

BsmtCond
TA      1311
Gd        65
Fa        45
None      37
Po         2
Name: count, dtype: int64

In [133]:
dataset['BsmtExposure'] = dataset['BsmtExposure'].fillna('None')

In [134]:
dataset['BsmtExposure'].isnull().value_counts(dropna=False)

BsmtExposure
False    1460
Name: count, dtype: int64

In [135]:
dataset['BsmtExposure'].value_counts()

BsmtExposure
No      953
Av      221
Gd      134
Mn      114
None     38
Name: count, dtype: int64

In [136]:
dataset['BsmtFinType2'] = dataset['BsmtFinType2'].fillna('None')

In [137]:
dataset['BsmtFinType2'].isnull().value_counts(dropna=False)

BsmtFinType2
False    1460
Name: count, dtype: int64

In [138]:
dataset['BsmtFinType2'].value_counts()

BsmtFinType2
Unf     1256
Rec       54
LwQ       46
None      38
BLQ       33
ALQ       19
GLQ       14
Name: count, dtype: int64

In [139]:
dataset['GarageQual'] = dataset['GarageQual'].fillna('None')

In [140]:
dataset['GarageQual'].isnull().value_counts(dropna=False)

GarageQual
False    1460
Name: count, dtype: int64

In [141]:
dataset['GarageQual'].value_counts()

GarageQual
TA      1311
None      81
Fa        48
Gd        14
Ex         3
Po         3
Name: count, dtype: int64

In [142]:
dataset['GarageCond'] = dataset['GarageCond'].fillna('None')

In [143]:
dataset['GarageCond'].isnull().value_counts(dropna=False)

GarageCond
False    1460
Name: count, dtype: int64

In [144]:
dataset['GarageCond'].value_counts()

GarageCond
TA      1326
None      81
Fa        35
Gd         9
Po         7
Ex         2
Name: count, dtype: int64

In [145]:
dataset['GarageFinish'] = dataset['GarageFinish'].fillna('None')

In [146]:
dataset['GarageFinish'].isnull().value_counts(dropna=False)

GarageFinish
False    1460
Name: count, dtype: int64

In [147]:
dataset['GarageFinish'].value_counts()

GarageFinish
Unf     605
RFn     422
Fin     352
None     81
Name: count, dtype: int64

In [148]:
dataset['GarageYrBlt'] = dataset['GarageYrBlt'].fillna(0)

In [149]:
dataset['GarageYrBlt'].isnull().value_counts(dropna=False)

GarageYrBlt
False    1460
Name: count, dtype: int64

In [150]:
dataset['GarageYrBlt'].value_counts()

GarageYrBlt
0.0       81
2005.0    65
2006.0    59
2004.0    53
2003.0    50
2007.0    49
1977.0    35
1998.0    31
1999.0    30
1976.0    29
2008.0    29
2000.0    27
2002.0    26
1968.0    26
1950.0    24
1993.0    22
1965.0    21
1962.0    21
1958.0    21
1966.0    21
2009.0    21
2001.0    20
1970.0    20
1957.0    20
1996.0    20
1960.0    19
1954.0    19
1997.0    19
1978.0    19
1995.0    18
1964.0    18
1974.0    18
1994.0    18
1959.0    17
1963.0    16
1990.0    16
1956.0    16
1967.0    15
1979.0    15
1969.0    15
1980.0    15
1973.0    14
1920.0    14
1988.0    14
1972.0    14
1940.0    14
1961.0    13
1971.0    13
1955.0    13
1992.0    13
1953.0    12
1987.0    11
1948.0    11
1985.0    10
1981.0    10
1989.0    10
1925.0    10
1941.0    10
1939.0     9
1991.0     9
1975.0     9
1930.0     8
1984.0     8
1949.0     8
1983.0     7
1951.0     6
1926.0     6
1986.0     6
1936.0     5
1916.0     5
1922.0     5
1931.0     4
1935.0     4
1945.0     4
1982.0     4
1928.0     4


In [151]:
dataset['GarageType'] = dataset['GarageType'].fillna('None')

In [152]:
dataset['GarageType'].isnull().value_counts(dropna=False)

GarageType
False    1460
Name: count, dtype: int64

In [153]:
dataset['GarageType'].value_counts()

GarageType
Attchd     870
Detchd     387
BuiltIn     88
None        81
Basment     19
CarPort      9
2Types       6
Name: count, dtype: int64

In [154]:
dataset['FireplaceQu'] = dataset['FireplaceQu'].fillna('None')

In [155]:
dataset['FireplaceQu'].isnull().value_counts(dropna=False)

FireplaceQu
False    1460
Name: count, dtype: int64

In [156]:
dataset['FireplaceQu'].value_counts()

FireplaceQu
None    690
Gd      380
TA      313
Fa       33
Ex       24
Po       20
Name: count, dtype: int64

In [157]:
dataset['Fence'] = dataset['Fence'].fillna('None')

In [158]:
dataset['Fence'].isnull().value_counts(dropna=False)

Fence
False    1460
Name: count, dtype: int64

In [159]:
dataset['Fence'].value_counts()

Fence
None     1179
MnPrv     157
GdPrv      59
GdWo       54
MnWw       11
Name: count, dtype: int64

In [160]:
dataset['MiscFeature'] = dataset['MiscFeature'].fillna('None')

In [161]:
dataset['MiscFeature'].isnull().value_counts(dropna=False)

MiscFeature
False    1460
Name: count, dtype: int64

In [162]:
dataset['MiscFeature'].value_counts()

MiscFeature
None    1406
Shed      49
Gar2       2
Othr       2
TenC       1
Name: count, dtype: int64

In [163]:
dataset['Alley'] = dataset['Alley'].fillna('None')

In [164]:
dataset['Alley'].isnull().value_counts(dropna=False)

Alley
False    1460
Name: count, dtype: int64

In [165]:
dataset['Alley'].value_counts()

Alley
None    1369
Grvl      50
Pave      41
Name: count, dtype: int64

In [166]:
dataset['PoolQC'] = dataset['PoolQC'].fillna('None')

In [167]:
dataset['PoolQC'].isnull().value_counts(dropna=False)

PoolQC
False    1460
Name: count, dtype: int64

In [168]:
dataset['PoolQC'].value_counts()

PoolQC
None    1453
Gd         3
Ex         2
Fa         2
Name: count, dtype: int64

In [169]:
dataset['BsmtQual'] = dataset['BsmtQual'].fillna('None')

In [170]:
dataset['BsmtQual'].isnull().value_counts(dropna=False)

BsmtQual
False    1460
Name: count, dtype: int64

In [171]:
dataset['BsmtQual'].value_counts()

BsmtQual
TA      649
Gd      618
Ex      121
None     37
Fa       35
Name: count, dtype: int64

In [172]:
dataset.isnull().sum().sort_values(ascending=False)

LotFrontage      259
Id                 0
BedroomAbvGr       0
GarageYrBlt        0
GarageType         0
FireplaceQu        0
Fireplaces         0
Functional         0
TotRmsAbvGrd       0
KitchenQual        0
KitchenAbvGr       0
HalfBath           0
GarageCars         0
FullBath           0
BsmtHalfBath       0
BsmtFullBath       0
GrLivArea          0
LowQualFinSF       0
2ndFlrSF           0
1stFlrSF           0
GarageFinish       0
GarageArea         0
CentralAir         0
PoolQC             0
SaleCondition      0
SaleType           0
YrSold             0
MoSold             0
MiscVal            0
MiscFeature        0
Fence              0
PoolArea           0
GarageQual         0
ScreenPorch        0
3SsnPorch          0
EnclosedPorch      0
OpenPorchSF        0
WoodDeckSF         0
PavedDrive         0
GarageCond         0
Electrical         0
HeatingQC          0
MSSubClass         0
LandSlope          0
OverallCond        0
OverallQual        0
HouseStyle         0
BldgType     

In [173]:
dataset.groupby('Neighborhood')['LotFrontage'].agg(['count', 'mean', 'median', 'min', 'max'])

,count,mean,median,min,max
Neighborhood,,,,,
Blmngtn,14,47.142857,43.0,43.0,53.0
Blueste,2,24.000000,24.0,24.0,24.0
BrDale,16,21.562500,21.0,21.0,24.0
BrkSide,51,57.509804,52.0,50.0,144.0
ClearCr,13,83.461538,80.0,62.0,138.0
CollgCr,126,71.682540,70.0,36.0,122.0
Crawfor,41,71.804878,74.0,40.0,130.0
Edwards,92,68.217391,65.5,24.0,313.0
Gilbert,49,79.877551,65.0,42.0,182.0


In [174]:
dataset['NeighborhoodMedian'] = dataset.groupby('Neighborhood')['LotFrontage'].transform('median')

In [175]:
dataset[['Neighborhood', 'LotFrontage', 'NeighborhoodMedian']].head(20)

,Neighborhood,LotFrontage,NeighborhoodMedian
0,CollgCr,65.0,70.0
1,Veenker,80.0,68.0
2,CollgCr,68.0,70.0
3,Crawfor,60.0,74.0
4,NoRidge,84.0,91.0
5,Mitchel,85.0,73.0
6,Somerst,75.0,73.5
7,NWAmes,NaN,80.0
8,OldTown,51.0,60.0
9,BrkSide,50.0,52.0


In [176]:
dataset[dataset['LotFrontage'].isnull()][['Neighborhood', 'LotFrontage', 'NeighborhoodMedian']].head(20)

,Neighborhood,LotFrontage,NeighborhoodMedian
7,NWAmes,NaN,80.0
12,Sawyer,NaN,71.0
14,NAmes,NaN,73.0
16,NAmes,NaN,73.0
24,Sawyer,NaN,71.0
31,Sawyer,NaN,71.0
42,SawyerW,NaN,66.5
43,CollgCr,NaN,70.0
50,Gilbert,NaN,65.0
64,CollgCr,NaN,70.0


In [177]:
dataset['LotFrontage'] = dataset['LotFrontage'].fillna(dataset['NeighborhoodMedian'])

In [178]:
dataset['LotFrontage'].isnull().sum()

np.int64(0)

In [179]:
dataset = dataset.drop('NeighborhoodMedian', axis=1)

In [180]:
'NeighborhoodMedian' in dataset.columns

False

In [181]:
dataset.loc[[7, 12, 14], ['Neighborhood',	'LotFrontage']]

,Neighborhood,LotFrontage
7,NWAmes,80.0
12,Sawyer,71.0
14,NAmes,73.0


In [182]:
dataset.isnull().sum().sort_values(ascending=False)

Id               0
CentralAir       0
GarageYrBlt      0
GarageType       0
FireplaceQu      0
Fireplaces       0
Functional       0
TotRmsAbvGrd     0
KitchenQual      0
KitchenAbvGr     0
BedroomAbvGr     0
HalfBath         0
FullBath         0
BsmtHalfBath     0
BsmtFullBath     0
GrLivArea        0
LowQualFinSF     0
2ndFlrSF         0
1stFlrSF         0
GarageFinish     0
GarageCars       0
GarageArea       0
PoolQC           0
SaleCondition    0
SaleType         0
YrSold           0
MoSold           0
MiscVal          0
MiscFeature      0
Fence            0
PoolArea         0
GarageQual       0
ScreenPorch      0
3SsnPorch        0
EnclosedPorch    0
OpenPorchSF      0
WoodDeckSF       0
PavedDrive       0
GarageCond       0
Electrical       0
HeatingQC        0
MSSubClass       0
Heating          0
OverallCond      0
OverallQual      0
HouseStyle       0
BldgType         0
Condition2       0
Condition1       0
Neighborhood     0
LandSlope        0
LotConfig        0
Utilities   

In [183]:
missing_values = dataset.isnull().sum()
missing_values[missing_values > 0].sum()

np.int64(0)

In [184]:
dataset[['Id', 'SalePrice']].head(20)

,Id,SalePrice
0,1,208500
1,2,181500
2,3,223500
3,4,140000
4,5,250000
5,6,143000
6,7,307000
7,8,200000
8,9,129900
9,10,118000


In [185]:
dataset['Id'].nunique()

1460

In [186]:
dataset[['Id', 'SalePrice']].corr()

,Id,SalePrice
Id,1.000000,-0.021917
SalePrice,-0.021917,1.000000


In [187]:
dataset = dataset.drop('Id', axis=1)

In [188]:
dataset

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,60,RL,65.0,8450,Pave,None,Reg,Lvl,AllPub,Inside,...,0,None,None,None,0,2,2008,WD,Normal,208500
1,20,RL,80.0,9600,Pave,None,Reg,Lvl,AllPub,FR2,...,0,None,None,None,0,5,2007,WD,Normal,181500
2,60,RL,68.0,11250,Pave,None,IR1,Lvl,AllPub,Inside,...,0,None,None,None,0,9,2008,WD,Normal,223500
3,70,RL,60.0,9550,Pave,None,IR1,Lvl,AllPub,Corner,...,0,None,None,None,0,2,2006,WD,Abnorml,140000
4,60,RL,84.0,14260,Pave,None,IR1,Lvl,AllPub,FR2,...,0,None,None,None,0,12,2008,WD,Normal,250000
5,50,RL,85.0,14115,Pave,None,IR1,Lvl,AllPub,Inside,...,0,None,MnPrv,Shed,700,10,2009,WD,Normal,143000
6,20,RL,75.0,10084,Pave,None,Reg,Lvl,AllPub,Inside,...,0,None,None,None,0,8,2007,WD,Normal,307000
7,60,RL,80.0,10382,Pave,None,IR1,Lvl,AllPub,Corner,...,0,None,None,Shed,350,11,2009,WD,Normal,200000
8,50,RM,51.0,6120,Pave,None,Reg,Lvl,AllPub,Inside,...,0,None,None,None,0,4,2008,WD,Abnorml,129900
9,190,RL,50.0,7420,Pave,None,Reg,Lvl,AllPub,Corner,...,0,None,None,None,0,1,2008,WD,Normal,118000


In [189]:
categorical_features = dataset.select_dtypes(include='str').columns

In [190]:
categorical_features

Index(['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',
       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
       'SaleType', 'SaleCondition'],
      dtype='str')

In [191]:
numerical_features = dataset.select_dtypes('number').columns

In [192]:
numerical_features

Index(['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond',
       'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2',
       'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF',
       'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath',
       'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces',
       'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF',
       'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal',
       'MoSold', 'YrSold', 'SalePrice'],
      dtype='str')

In [193]:
dataset.groupby('HouseStyle')['SalePrice'].agg(['count', 'mean', 'median', 'max', 'min']).sort_values('mean', ascending=False)

,count,mean,median,max,min
HouseStyle,,,,,
2.5Fin,8,220000.000000,194000.0,475000,104000
2Story,445,210051.764045,190000.0,755000,40000
1Story,726,175985.477961,154750.0,611657,34900
SLvl,65,166703.384615,164500.0,345000,91000
2.5Unf,11,157354.545455,133900.0,325000,101000
1.5Fin,154,143116.740260,132000.0,410000,37900
SFoyer,37,135074.486486,135960.0,206300,75500
1.5Unf,14,110150.000000,111250.0,139400,76000


In [194]:
house_style_mean = dataset.groupby('HouseStyle')['SalePrice'].mean()

In [195]:
house_style_mean.max() - house_style_mean.min()

np.float64(109850.0)

In [196]:
neighborhood_mean = dataset.groupby('Neighborhood')['SalePrice'].mean()

In [197]:
neighborhood_mean.max() - neighborhood_mean.min()

np.float64(236718.84648493544)

In [198]:
neighborhood_mean.sort_values(ascending=False)

Neighborhood
NoRidge    335295.317073
NridgHt    316270.623377
StoneBr    310499.000000
Timber     242247.447368
Veenker    238772.727273
Somerst    225379.837209
ClearCr    212565.428571
Crawfor    210624.725490
CollgCr    197965.773333
Blmngtn    194870.882353
Gilbert    192854.506329
NWAmes     189050.068493
SawyerW    186555.796610
Mitchel    156270.122449
NAmes      145847.080000
NPkVill    142694.444444
SWISU      142591.360000
Blueste    137500.000000
Sawyer     136793.135135
OldTown    128225.300885
Edwards    128219.700000
BrkSide    124834.051724
BrDale     104493.750000
IDOTRR     100123.783784
MeadowV     98576.470588
Name: SalePrice, dtype: float64

In [199]:
features_mean = {}
for feature in categorical_features:
    group_mean = (dataset.groupby(feature)['SalePrice'].mean())
    difference = group_mean.max() - group_mean.min()
    features_mean[feature] = difference

In [200]:
features_mean

{'MSZoning': np.float64(139486.06153846154),
 'Street': np.float64(50940.03851444292),
 'Alley': np.float64(61233.05148283417),
 'LotShape': np.float64(75078.54747528015),
 'LandContour': np.float64(88429.86063492062),
 'Utilities': np.float64(43450.95681973954),
 'LotConfig': np.float64(46916.569492759474),
 'LandSlope': np.float64(24422.431203384185),
 'Neighborhood': np.float64(236718.84648493544),
 'Condition1': np.float64(90783.33333333334),
 'Condition2': np.float64(228250.0),
 'BldgType': np.float64(57331.54931253304),
 'HouseStyle': np.float64(109850.0),
 'RoofStyle': np.float64(76090.90909090909),
 'RoofMatl': np.float64(253250.0),
 'Exterior1st': np.float64(191000.0),
 'Exterior2nd': np.float64(214000.0),
 'MasVnrType': np.float64(120156.5794871795),
 'ExterQual': np.float64(279375.7472527473),
 'ExterCond': np.float64(124833.33333333334),
 'Foundation': np.float64(117864.81704018547),
 'BsmtQual': np.float64(221388.14943042217),
 'BsmtCond': np.float64(149599.9076923077),
 '

In [201]:
type(features_mean)

dict

In [202]:
pd.Series(features_mean).sort_values(ascending=False)

PoolQC           309595.336545
ExterQual        279375.747253
RoofMatl         253250.000000
Neighborhood     236718.846485
Condition2       228250.000000
KitchenQual      222989.464872
BsmtQual         221388.149430
Exterior2nd      214000.000000
FireplaceQu      207948.350000
Exterior1st      191000.000000
SaleCondition    168166.752000
MiscFeature      156000.000000
SaleType         155095.418033
GarageType       151434.454686
BsmtExposure     149751.463865
BsmtCond         149599.907692
GarageQual       140833.333333
MSZoning         139486.061538
GarageFinish     136735.406390
BsmtFinType1     129760.828204
HeatingQC        127914.429150
ExterCond        124833.333333
MasVnrType       120156.579487
Electrical       119810.637453
Foundation       117864.817040
HouseStyle       109850.000000
Heating          109521.195378
BsmtFinType2      99595.868421
Functional        97629.147059
Condition1        90783.333333
LandContour       88429.860635
GarageCond        84568.451344
CentralA

In [203]:
dataset.groupby('PoolQC')['SalePrice'].agg(['count', 'mean'])

,count,mean
PoolQC,,
Ex,2,490000.000000
Fa,2,215500.000000
Gd,3,201990.000000
None,1453,180404.663455


In [204]:
dataset.groupby('ExterQual')['SalePrice'].agg(['count', 'mean'])

,count,mean
ExterQual,,
Ex,52,367360.961538
Fa,14,87985.214286
Gd,488,231633.510246
TA,906,144341.313466


In [205]:
groups_lowest_counts = {}
for feature in categorical_features:
    lowest_count = dataset.groupby(feature).size().min()
    groups_lowest_counts[feature] = lowest_count

In [206]:
groups_lowest_counts

{'MSZoning': np.int64(10),
 'Street': np.int64(6),
 'Alley': np.int64(41),
 'LotShape': np.int64(10),
 'LandContour': np.int64(36),
 'Utilities': np.int64(1),
 'LotConfig': np.int64(4),
 'LandSlope': np.int64(13),
 'Neighborhood': np.int64(2),
 'Condition1': np.int64(2),
 'Condition2': np.int64(1),
 'BldgType': np.int64(31),
 'HouseStyle': np.int64(8),
 'RoofStyle': np.int64(2),
 'RoofMatl': np.int64(1),
 'Exterior1st': np.int64(1),
 'Exterior2nd': np.int64(1),
 'MasVnrType': np.int64(15),
 'ExterQual': np.int64(14),
 'ExterCond': np.int64(1),
 'Foundation': np.int64(3),
 'BsmtQual': np.int64(35),
 'BsmtCond': np.int64(2),
 'BsmtExposure': np.int64(38),
 'BsmtFinType1': np.int64(37),
 'BsmtFinType2': np.int64(14),
 'Heating': np.int64(1),
 'HeatingQC': np.int64(1),
 'CentralAir': np.int64(95),
 'Electrical': np.int64(1),
 'KitchenQual': np.int64(39),
 'Functional': np.int64(1),
 'FireplaceQu': np.int64(20),
 'GarageType': np.int64(6),
 'GarageFinish': np.int64(81),
 'GarageQual': np.in

In [207]:
pd.Series(groups_lowest_counts).sort_values(ascending=False)

CentralAir       95
GarageFinish     81
Alley            41
KitchenQual      39
BsmtExposure     38
BsmtFinType1     37
LandContour      36
BsmtQual         35
BldgType         31
PavedDrive       30
FireplaceQu      20
MasVnrType       15
BsmtFinType2     14
ExterQual        14
LandSlope        13
Fence            11
MSZoning         10
LotShape         10
HouseStyle        8
Street            6
GarageType        6
SaleCondition     4
LotConfig         4
Foundation        3
GarageQual        3
Neighborhood      2
SaleType          2
PoolQC            2
GarageCond        2
Condition1        2
RoofStyle         2
BsmtCond          2
Exterior1st       1
Functional        1
Electrical        1
Condition2        1
HeatingQC         1
Heating           1
Utilities         1
RoofMatl          1
MiscFeature       1
ExterCond         1
Exterior2nd       1
dtype: int64

In [208]:
dataset.groupby('RoofMatl')['SalePrice'].agg(['count', 'mean']).sort_values('count')

,count,mean
RoofMatl,,
ClyTile,1,160000.000000
Membran,1,241500.000000
Metal,1,180000.000000
Roll,1,137000.000000
WdShake,5,241400.000000
WdShngl,6,390250.000000
Tar&Grv,11,185406.363636
CompShg,1434,179803.679219


In [209]:
dataset[dataset['ExterQual'] == 'Fa']['SalePrice'].sort_values()

533      39300
705      55000
812      55993
636      60000
375      61000
1061     81000
341      82000
1323     82500
88       85000
821      93000
1370    105000
52      110000
1266    122000
635     200000
Name: SalePrice, dtype: int64

In [210]:
groups = dataset.groupby('HouseStyle')['SalePrice']

In [211]:
group_sale_price = []
for group_name, sales_prices in groups:
    group_sale_price.append(sales_prices)

In [212]:
len(group_sale_price)

8

In [213]:
group_sale_price

[5       143000
 8       129900
 46      239686
 51      114500
 69      225000
 77      127000
 88       85000
 104     169500
 108     115000
 110     136900
 121     100000
 125      84500
 142     166000
 149     115000
 155      79000
 165     127500
 170     128500
 177     172500
 183     200000
 184     127000
 187     135000
 202     112000
 204     110000
 218     311500
 239     113000
 242      79000
 249     277000
 263     130000
 275     205000
 286     159000
 292     131000
 296     152000
 307      89500
 312     119900
 335     228950
 345     140200
 352      95000
 354     140000
 361     145000
 371     134432
 380     127000
 386      81000
 390     119000
 394     109000
 406     115000
 418     126000
 431      79900
 436     116000
 439     110000
 442     162900
 448     119500
 449     120000
 459     110000
 488     160000
 491     133000
 497     184000
 522     159000
 545     229000
 546     210000
 557     108000
 563     185000
 568     316600
 575    

In [214]:
anova_results = f_oneway(*group_sale_price)

In [215]:
anova_results

F_onewayResult(statistic=np.float64(19.59500099598122), pvalue=np.float64(3.376776535118099e-25))

In [216]:
anova_f_scores = {} 
for feature in categorical_features:
    groups = dataset.groupby(feature)['SalePrice']
    
    group_sale_price = []
    for group_name, sale_prices in groups:
        group_sale_price.append(sale_prices)
    
    anova_results = f_oneway(*group_sale_price)
    anova_f_scores[feature] = anova_results.statistic

In [217]:
pd.Series(anova_f_scores).sort_values(ascending=False)

ExterQual        443.334831
KitchenQual      407.806352
BsmtQual         316.148635
GarageFinish     213.867028
FireplaceQu      121.075121
MasVnrType       112.882739
Foundation       100.253851
CentralAir        98.305344
HeatingQC         88.394462
GarageType        80.379992
Neighborhood      71.784865
BsmtFinType1      64.688200
BsmtExposure      63.939761
SaleCondition     45.578428
MSZoning          43.840282
PavedDrive        42.024179
LotShape          40.132852
SaleType          28.863054
GarageQual        25.776093
GarageCond        25.750153
Electrical        23.074438
BsmtCond          19.708139
HouseStyle        19.595001
Exterior1st       18.611743
RoofStyle         17.805497
Exterior2nd       17.500840
Alley             15.176614
Fence             13.433276
BldgType          13.011077
LandContour       12.850188
PoolQC            10.509853
ExterCond          8.798714
LotConfig          7.809954
BsmtFinType2       7.565378
RoofMatl           6.727305
Condition1         6

In [218]:
dataset.groupby('ExterQual')['SalePrice'].count()

ExterQual
Ex     52
Fa     14
Gd    488
TA    906
Name: SalePrice, dtype: int64

In [219]:
dataset.groupby('KitchenQual')['SalePrice'].count()

KitchenQual
Ex    100
Fa     39
Gd    586
TA    735
Name: SalePrice, dtype: int64

In [220]:
dataset.groupby('ExterQual')['SalePrice'].mean()

ExterQual
Ex    367360.961538
Fa     87985.214286
Gd    231633.510246
TA    144341.313466
Name: SalePrice, dtype: float64

In [221]:
dataset.groupby('KitchenQual')['SalePrice'].mean()

KitchenQual
Ex    328554.670000
Fa    105565.205128
Gd    212116.023891
TA    139962.511565
Name: SalePrice, dtype: float64

In [222]:
pd.crosstab(dataset['KitchenQual'], dataset['ExterQual'])

ExterQual,Ex,Fa,Gd,TA
KitchenQual,,,,
Ex,42,0,47,11
Fa,0,8,0,31
Gd,8,0,408,170
TA,2,6,33,694


In [223]:
pd.crosstab(dataset['KitchenQual'], dataset['ExterQual'], normalize='index')

ExterQual,Ex,Fa,Gd,TA
KitchenQual,,,,
Ex,0.420000,0.000000,0.470000,0.110000
Fa,0.000000,0.205128,0.000000,0.794872
Gd,0.013652,0.000000,0.696246,0.290102
TA,0.002721,0.008163,0.044898,0.944218


In [224]:
table = pd.crosstab(dataset['KitchenQual'], dataset['ExterQual'])

In [225]:
table

ExterQual,Ex,Fa,Gd,TA
KitchenQual,,,,
Ex,42,0,47,11
Fa,0,8,0,31
Gd,8,0,408,170
TA,2,6,33,694


In [226]:
# chi_square
chi2, p, dof, expected = chi2_contingency(table)
print(f"chi_square: {chi2}")
print(f"p-value: {p}")
print("degress of freedom:", dof)

chi_square: 1312.2513151093713
p-value: 6.988389926465085e-277
degress of freedom: 9


In [227]:
table.shape

(4, 4)

In [228]:
# Calculating Cramér's V
math.sqrt(chi2 / (1460 * min(4-1, 4-1)))

0.5473579787253363

In [229]:
expected

array([[3.56164384e+00, 9.58904110e-01, 3.34246575e+01, 6.20547945e+01],
       [1.38904110e+00, 3.73972603e-01, 1.30356164e+01, 2.42013699e+01],
       [2.08712329e+01, 5.61917808e+00, 1.95868493e+02, 3.63641096e+02],
       [2.61780822e+01, 7.04794521e+00, 2.45671233e+02, 4.56102740e+02]])

In [230]:
dataset.groupby(['KitchenQual', 'ExterQual'])['SalePrice'].mean()

KitchenQual  ExterQual
Ex           Ex           384854.214286
             Gd           317257.234043
             TA           161863.636364
Fa           Fa            79849.125000
             TA           112201.612903
Gd           Ex           322174.125000
             Gd           226499.237745
             TA           172417.105882
TA           Ex           180750.000000
             Fa            98833.333333
             Gd           173162.848485
             TA           138621.861671
Name: SalePrice, dtype: float64

In [231]:
len(group_sale_price)

6

In [232]:
feature = 'ExterQual'
group = dataset.groupby(feature)['SalePrice']

In [233]:
group

In [234]:
results = []
for feature in categorical_features:
    results.append(dataset.groupby([feature])['SalePrice'].agg(['count', 'mean', 'median']).sort_values('mean', ascending=False))
results

[          count           mean    median
 MSZoning                                
 FV           65  214014.061538  205950.0
 RL         1151  191004.994787  174000.0
 RH           16  131558.375000  136500.0
 RM          218  126316.830275  120500.0
 C (all)      10   74528.000000   74700.0,
         count           mean    median
 Street                                
 Pave     1454  181130.538514  163000.0
 Grvl        6  130190.500000  114250.0,
        count           mean    median
 Alley                                
 None    1369  183452.131483  165000.0
 Pave      41  168000.585366  172500.0
 Grvl      50  122219.080000  119500.0,
           count           mean    median
 LotShape                                
 IR2          41  239833.365854  221000.0
 IR3          10  216036.500000  203570.0
 IR1         484  206101.665289  189000.0
 Reg         925  164754.818378  146000.0,
              count           mean    median
 LandContour                                
 HLS 

In [235]:
dataset[categorical_features].nunique()

MSZoning          5
Street            2
Alley             3
LotShape          4
LandContour       4
Utilities         2
LotConfig         5
LandSlope         3
Neighborhood     25
Condition1        9
Condition2        8
BldgType          5
HouseStyle        8
RoofStyle         6
RoofMatl          8
Exterior1st      15
Exterior2nd      16
MasVnrType        4
ExterQual         4
ExterCond         5
Foundation        6
BsmtQual          5
BsmtCond          5
BsmtExposure      5
BsmtFinType1      7
BsmtFinType2      7
Heating           6
HeatingQC         5
CentralAir        2
Electrical        5
KitchenQual       4
Functional        7
FireplaceQu       6
GarageType        7
GarageFinish      4
GarageQual        6
GarageCond        6
PavedDrive        3
PoolQC            4
Fence             5
MiscFeature       5
SaleType          9
SaleCondition     6
dtype: int64

In [236]:
X = dataset[['ExterQual']]
y = dataset['SalePrice']

In [237]:
X.shape

(1460, 1)

In [238]:
y.shape

(1460,)

In [239]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [240]:
X_train.shape

(1168, 1)

In [241]:
X_test.shape

(292, 1)

In [242]:
y_train.shape

(1168,)

In [243]:
y_test.shape

(292,)

In [244]:
encoder = OneHotEncoder()

In [245]:
X_train_encoded = encoder.fit_transform(X_train)

In [246]:
encoder

,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a SciPy sparse matrix/arrayin ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",True
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'error'
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide <encoder_infrequent

In [247]:
X_test_encoded = encoder.transform(X_test)

In [248]:
modelA = LinearRegression()

In [249]:
modelA.fit(X_train_encoded, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](4,)","[ 146287.97,-111838.42, 24924.93, -59374.48]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,2.059e+05
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,4


In [250]:
modelA_pred = modelA.predict(X_test_encoded)

In [251]:
modelA_mae = mean_absolute_error(y_test, modelA_pred)

In [252]:
modelA_mae

40003.945943276565

In [253]:
X_2 = dataset[['ExterQual', 'KitchenQual']]
y_2 = dataset['SalePrice']

In [254]:
X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(X_2, y_2, test_size=0.20, random_state=42)

In [255]:
encoder_2 = OneHotEncoder()

In [256]:
X_train_encoded_2 = encoder_2.fit_transform(X_train_2)

In [257]:
X_test_encoded_2 = encoder_2.transform(X_test_2)

In [258]:
modelB = LinearRegression()

In [259]:
modelB.fit(X_train_encoded_2, y_train_2)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](8,)","[ 81922.6 ,-64486.24, 16069.53,...,-58074.13, 4870.32,-31380.82]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,2.045e+05
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,8


In [260]:
modelB_pred = modelB.predict(X_test_encoded_2)

In [261]:
modelB_mae = mean_absolute_error(y_test_2, modelB_pred)

In [262]:
modelB_mae

38156.54920283288

In [263]:
kf = KFold(n_splits=5, random_state=42, shuffle=True)

In [264]:
for train_index, val_index in kf.split(X_train_2):
    print(f"Training length", len(train_index))
    print(f"Validation length", len(val_index))

Training length 934
Validation length 234
Training length 934
Validation length 234
Training length 934
Validation length 234
Training length 935
Validation length 233
Training length 935
Validation length 233


In [265]:
mae_values = []
for train_index, val_index in kf.split(X_train_2):
    X_train_fold = X_train_2.iloc[train_index]
    X_val_fold = X_train_2.iloc[val_index]

    y_train_fold = y_train_2.iloc[train_index]
    y_val_fold = y_train_2.iloc[val_index]
    
    encoder = OneHotEncoder()
    
    X_train_fold_encoded = encoder.fit_transform(X_train_fold)
    X_val_fold_encoded = encoder.transform(X_val_fold)
    
    modelC = LinearRegression()
    
    modelC.fit(X_train_fold_encoded, y_train_fold)
    
    modelC_pred = modelC.predict(X_val_fold_encoded)
    
    modelC_mae = mean_absolute_error(y_val_fold, modelC_pred)
    mae_values.append(modelC_mae)

In [266]:
mae_values

[37946.85647942454,
 37915.8090966761,
 39136.80430901418,
 36632.84629453827,
 35818.98974087559]

In [267]:
data = pd.Series(mae_values)

In [268]:
data.mean()

np.float64(37490.26118410573)

In [269]:
# Experiment 1(with 2 features)
# Pipeline
pipeline = Pipeline([
    ('encoder', OneHotEncoder()),
    ('modelD', LinearRegression())
])

In [270]:
# Cross Validation Scores
scores = cross_val_score(
    pipeline,
    X_train_2,
    y_train_2,
    cv=kf,
    scoring='neg_mean_absolute_error'
)

In [271]:
scores

array([-37946.85647942, -37915.80909668, -39136.80430901, -36632.84629454,
       -35818.98974088])

In [272]:
mae_scores = -scores

In [273]:
data_experiment_1 = pd.Series(mae_scores)

In [274]:
data_experiment_1.mean()

np.float64(37490.26118410573)

In [275]:
# Experiment 2(with 3 features)
X_3 = dataset[['ExterQual', 'KitchenQual', 'BsmtQual']]
y_3 = dataset['SalePrice']

In [276]:
X_train_3, X_test_3, y_train_3, y_test_3 = train_test_split(X_3, y_3, test_size=0.20, random_state=42)

In [277]:
# Pipeline
pipeline_2 = Pipeline([
    ('encoder', OneHotEncoder()),
    ('modelE', LinearRegression())
    
])

In [278]:
scores_experiment_2 = cross_val_score(
    pipeline_2,
    X_train_3,
    y_train_3,
    cv=kf,
    scoring='neg_mean_absolute_error'
)

In [279]:
scores_experiment_2

array([-35577.59619007, -34545.63930744, -36917.83155896, -32475.07255385,
       -32865.84989946])

In [280]:
mae_scores_experiment_2 = -scores_experiment_2

In [281]:
data_experiment_2 = pd.Series(mae_scores_experiment_2)

In [282]:
data_experiment_2.mean()

np.float64(34476.39790195465)

In [283]:
# Experiment 3(with 4 features)
X_4 = dataset[['ExterQual', 'KitchenQual', 'BsmtQual', 'GarageFinish']]
y_4 = dataset['SalePrice']

In [284]:
X_train_4, X_test_4, y_train_4, y_test_4 = train_test_split(X_4, y_4, test_size=0.20, random_state=42)

In [285]:
pipeline_3 = Pipeline([
    ('encoder', OneHotEncoder()),
    ('modelF', LinearRegression())
])

In [286]:
scores_experiment_3 = cross_val_score(
    pipeline_3,
    X_train_4,
    y_train_4,
    cv=kf,
    scoring='neg_mean_absolute_error'
)

In [287]:
scores_experiment_3

array([-33907.20012378, -32629.42937229, -34630.18236278, -30579.16399933,
       -31600.7475634 ])

In [288]:
mae_scores_experiment_3 = -scores_experiment_3

In [289]:
mae_scores_experiment_3

array([33907.20012378, 32629.42937229, 34630.18236278, 30579.16399933,
       31600.7475634 ])

In [290]:
data_experiment_3 = pd.Series(mae_scores_experiment_3)

In [291]:
data_experiment_3

0    33907.200124
1    32629.429372
2    34630.182363
3    30579.163999
4    31600.747563
dtype: float64

In [292]:
data_experiment_3.mean()

np.float64(32669.344684314845)

In [293]:
print(f"std for mae with 2 features", data_experiment_1.std())
print(f"std for mae with 3 features", data_experiment_2.std())
print(f"std for mae with 4 features", data_experiment_3.std())

std for mae with 2 features 1287.3405805761035
std for mae with 3 features 1855.8824289855997
std for mae with 4 features 1649.9635469564191


In [294]:
# Experiment 5(with 5 features)
X_5 = dataset[['ExterQual', 'KitchenQual', 'BsmtQual', 'GarageFinish', 'GarageType']]
y_5 = dataset['SalePrice']

In [295]:
X_train_5, X_test_5, y_train_5, y_test_5 = train_test_split(X_5, y_5, test_size=0.20, random_state=42)

In [296]:
pipeline_4 = Pipeline([
    ('encoder', OneHotEncoder()),
    ('modelE', LinearRegression())
])

In [297]:
scores_experiment_4 = cross_val_score(
    pipeline_4,
    X_train_5,
    y_train_5,
    cv=kf,
    scoring='neg_mean_absolute_error'
)

In [298]:
scores_experiment_4

array([-32923.72775626, -31846.84837264, -34141.74070235, -31065.51787152,
       -30645.33210437])

In [299]:
mae_scores_experiment_4 = -scores_experiment_4

In [300]:
mae_scores_experiment_4

array([32923.72775626, 31846.84837264, 34141.74070235, 31065.51787152,
       30645.33210437])

In [301]:
data_experiment_4 = pd.Series(mae_scores_experiment_4)

In [302]:
data_experiment_4.mean()

np.float64(32124.633361426124)

In [303]:
data_experiment_4.std()

np.float64(1422.541405965847)

In [304]:
# Experiment 6(with 6 features)
X_6 = dataset[['ExterQual', 'KitchenQual', 'BsmtQual', 'GarageFinish', 'GarageType', 'FireplaceQu']]
y_6 = dataset['SalePrice']

In [305]:
X_train_6, X_test_6, y_train_6, y_test_6 = train_test_split(X_6, y_6, test_size=0.20, random_state=42)

In [306]:
pipeline_experiment_6 = Pipeline([
    ('encoder', OneHotEncoder()),
    ('modelF', LinearRegression())
])

In [307]:
scores_experiment_6 = cross_val_score(
    pipeline_experiment_6,
    X_train_6,
    y_train_6,
    cv=kf,
    scoring='neg_mean_absolute_error'
)

In [308]:
scores_experiment_6

array([-31455.61240437, -29077.07957352, -32908.01903086, -29379.82761864,
       -28029.83558   ])

In [309]:
mae_scores_experiment_6 = -scores_experiment_6

In [310]:
mae_scores_experiment_6

array([31455.61240437, 29077.07957352, 32908.01903086, 29379.82761864,
       28029.83558   ])

In [311]:
data_experiment_6 = pd.Series(mae_scores_experiment_6)

In [312]:
data_experiment_6.mean()

np.float64(30170.074841477104)

In [313]:
data_experiment_6.std()

np.float64(1971.591909798857)

In [314]:
# Experiment 7(with 7 features)
X_7 = dataset[['ExterQual', 'KitchenQual', 'BsmtQual', 'GarageFinish', 'GarageType', 'FireplaceQu', 'MasVnrType']]
y_7 = dataset['SalePrice']

In [315]:
X_train_7, X_test_7, y_train_7, y_test_7 = train_test_split(X_7, y_7, test_size=0.20, random_state=42)

In [316]:
pipeline_experiment_7 = Pipeline([
    ('encoder', OneHotEncoder()),
    ('model', LinearRegression())
])

In [317]:
scores_experiment_7 = cross_val_score(
    pipeline_experiment_7,
    X_train_7,
    y_train_7,
    cv=kf,
    scoring='neg_mean_absolute_error'
)

In [318]:
scores_experiment_7

array([-31478.07565903, -29008.12948744, -32467.66665127, -29520.24934174,
       -27884.64647271])

In [319]:
mae_scores_experiment_7 = -scores_experiment_7

In [320]:
mae_scores_experiment_7

array([31478.07565903, 29008.12948744, 32467.66665127, 29520.24934174,
       27884.64647271])

In [321]:
data_experiment_7 = pd.Series(mae_scores_experiment_7)

In [322]:
data_experiment_7

0    31478.075659
1    29008.129487
2    32467.666651
3    29520.249342
4    27884.646473
dtype: float64

In [323]:
data_experiment_7.mean()

np.float64(30071.753522434847)

In [324]:
data_experiment_7.std()

np.float64(1866.6167167421852)

In [325]:
# Experiment 1 with numerical feature(1 feature)
X_num_1 = dataset[['OverallQual']]
y_num_1 = dataset['SalePrice']

In [326]:
X_train_num_1, X_test_num_1, y_train_num_1, y_test_num_1 = train_test_split(X_num_1, y_num_1, test_size=0.20, random_state=42)

In [327]:
pipeline_num_1 = Pipeline([
    ('model', LinearRegression())
])

In [328]:
scores_num_1 = cross_val_score(
    pipeline_num_1,
    X_train_num_1,
    y_train_num_1,
    cv=kf,
    scoring='neg_mean_absolute_error'
)

In [329]:
scores_num_1

array([-32927.24129335, -33788.98026162, -35121.65566421, -35218.41347433,
       -31301.42503828])

In [330]:
scores_num_positive = -scores_num_1

In [331]:
data_num_1 = pd.Series(scores_num_positive)

In [332]:
data_num_1

0    32927.241293
1    33788.980262
2    35121.655664
3    35218.413474
4    31301.425038
dtype: float64

In [333]:
mae_num_1 = data_num_1.mean()

In [334]:
mae_num_1

np.float64(33671.54314635731)

In [335]:
data_num_1.std()

np.float64(1634.0800696933215)

In [336]:
# Experiment 2 with numerical feature(2 feature)
X_num_2 = dataset[['OverallQual', 'GrLivArea']]
y_num_2 = dataset['SalePrice']

In [337]:
X_train_num_2, X_test_num_2, y_train_num_2, y_test_num_2 = train_test_split(X_num_2, y_num_2, test_size=0.20, random_state=42)

In [338]:
pipeline_num_2 = Pipeline([
    ('model', LinearRegression())
])

In [339]:
scores_num_2 = cross_val_score(
    pipeline_num_2,
    X_train_num_2,
    y_train_num_2,
    cv=kf,
    scoring='neg_mean_absolute_error'
)

In [340]:
scores_num_2

array([-27283.77237853, -29905.64508322, -29622.93645248, -29378.5944704 ,
       -27161.35020875])

In [341]:
scores_num_2_positive = -scores_num_2

In [342]:
scores_num_2_positive

array([27283.77237853, 29905.64508322, 29622.93645248, 29378.5944704 ,
       27161.35020875])

In [343]:
data_num_2 = pd.Series(scores_num_2_positive)

In [344]:
data_num_2.mean()

np.float64(28670.45971867768)

In [345]:
data_num_2.std()

np.float64(1335.5394913739012)

In [346]:
# Experiment 3 with numerical feature(3 feature)
X_num_3 = dataset[['OverallQual', 'GrLivArea', 'GarageCars']]
y_num_3 = dataset['SalePrice']

In [347]:
X_train_num_3, X_test_num_3, y_train_num_3, y_test_num_3 = train_test_split(X_num_3, y_num_3, test_size=0.20, random_state=42)

In [348]:
pipeline_num_3 = Pipeline([
    ('model', LinearRegression())
])

In [349]:
scores_num_3 = cross_val_score(
    pipeline_num_3,
    X_train_num_3,
    y_train_num_3,
    cv=kf,
    scoring='neg_mean_absolute_error'
)

In [350]:
scores_num_3_positive = -scores_num_3

In [351]:
scores_num_3_positive

array([26556.38937345, 28905.84492607, 28460.8548299 , 26669.22455086,
       25793.34496968])

In [352]:
data_num_3 = pd.Series(scores_num_3_positive)

In [353]:
data_num_3.mean()

np.float64(27277.13172999234)

In [354]:
data_num_3.std()

np.float64(1336.464635274692)

In [355]:
# Experiment 4 with numerical feature(4 feature)
X_num_4 = dataset[['OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea']]
y_num_4 = dataset['SalePrice']

In [356]:
X_train_num_4, X_test_num_4, y_train_num_4, y_test_num_4 = train_test_split(X_num_4, y_num_4, test_size=0.20, random_state=42)

In [357]:
pipeline_num_4 = Pipeline([
    ('model', LinearRegression())
])

In [358]:
scores_num_4 = cross_val_score(
    pipeline_num_4,
    X_train_num_4,
    y_train_num_4,
    cv=kf,
    scoring='neg_mean_absolute_error'
)

In [359]:
scores_num_4_positive = -scores_num_4

In [360]:
scores_num_4_positive

array([26357.29179571, 28432.1490222 , 28055.40865113, 26275.87398555,
       25429.88739042])

In [361]:
scores_num_4_positive.mean()

np.float64(26910.122169003072)

In [362]:
scores_num_4_positive.std()

np.float64(1142.5615123519458)

In [363]:
# Experiment 5 with numerical feature(5 feature)
X_num_5 = dataset[['OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea', 'TotalBsmtSF']]
y_num_5 = dataset['SalePrice']

In [364]:
X_train_num_5, X_test_num_5, y_train_num_5, y_test_num_5 = train_test_split(X_num_5, y_num_5, test_size=0.20, random_state=42)

In [365]:
pipeline_num_5 = Pipeline([
    ('model', LinearRegression())
])

In [366]:
scores_num_5 = cross_val_score(
    pipeline_num_5,
    X_train_num_5,
    y_train_num_5,
    cv=kf,
    scoring='neg_mean_absolute_error'
)

In [367]:
scores_num_5_positive = -scores_num_5

In [368]:
scores_num_5_positive

array([24453.53876723, 27114.34710962, 26608.94217702, 24475.07551807,
       23523.94405348])

In [369]:
scores_num_5_positive.mean()

np.float64(25235.169525083376)

In [370]:
scores_num_5_positive.std()

np.float64(1380.9809447798016)

In [371]:
# Experiment 6 with numerical feature(6 feature)
X_num_6 = dataset[['OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea', 'TotalBsmtSF', 'Fireplaces']]
y_num_6 = dataset['SalePrice']

In [372]:
X_train_num_6, X_test_num_6, y_train_num_6, y_test_num_6 = train_test_split(X_num_6, y_num_6, test_size=0.20, random_state=42)

In [373]:
pipeline_num_6 = Pipeline([
    ('model', LinearRegression())
])

In [374]:
scores_num_6 = cross_val_score(
    pipeline_num_6,
    X_train_num_6,
    y_train_num_6,
    cv=kf,
    scoring='neg_mean_absolute_error'
)

In [375]:
scores_num_6_positive = -scores_num_6

In [376]:
scores_num_6_positive

array([24255.26138556, 26804.32765829, 26333.29579001, 24981.81957371,
       23093.59491315])

In [377]:
scores_num_6_positive.mean()

np.float64(25093.6598641435)

In [378]:
scores_num_6_positive.std()

np.float64(1354.8993762072917)

In [379]:
# Experiment 7 with numerical feature(7 feature)
X_num_7 = dataset[['OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea', 'TotalBsmtSF', 'Fireplaces', 'YearBuilt']]
y_num_7 = dataset['SalePrice']

In [380]:
X_train_num_7, X_test_num_7, y_train_num_7, y_test_num_7 = train_test_split(X_num_7, y_num_7, test_size=0.20, random_state=42)

In [381]:
pipeline_num_7 = Pipeline([
    ('model', LinearRegression())
])

In [382]:
scores_num_7 = cross_val_score(
    pipeline_num_7,
    X_train_num_7,
    y_train_num_7,
    cv=kf,
    scoring='neg_mean_absolute_error'
)

In [383]:
scores_num_7_positive = -scores_num_7

In [384]:
scores_num_7_positive

array([23867.73166943, 25990.24301532, 25051.16998777, 24101.89511806,
       22631.58319117])

In [385]:
scores_num_7_positive.mean()

np.float64(24328.524596349074)

In [386]:
scores_num_7_positive.std()

np.float64(1133.739444513933)

In [387]:
# Categorical baseline:
# Mean CV MAE ≈ $30,071.75

# Numerical baseline:
# Mean CV MAE ≈ $24,328.52

In [388]:
X_mixed = dataset[['OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea', 'TotalBsmtSF', 'YearBuilt', 'Fireplaces', 'ExterQual', 'KitchenQual', 'BsmtQual', 'GarageFinish', 'GarageType', 'FireplaceQu', 'MasVnrType']]
y_mixed = dataset['SalePrice']

In [389]:
X_mixed.shape

(1460, 14)

In [390]:
X_mixed.dtypes

OverallQual     int64
GrLivArea       int64
GarageCars      int64
GarageArea      int64
TotalBsmtSF     int64
YearBuilt       int64
Fireplaces      int64
ExterQual         str
KitchenQual       str
BsmtQual          str
GarageFinish      str
GarageType        str
FireplaceQu       str
MasVnrType        str
dtype: object

In [391]:
X_mixed_train, X_mixed_test, y_mixed_train, y_mixed_test = train_test_split(X_mixed, y_mixed, test_size=0.20, random_state=42)

In [392]:
numerical_features = ['OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea', 'TotalBsmtSF', 'YearBuilt', 'Fireplaces']
categorical_features = ['ExterQual', 'KitchenQual', 'BsmtQual', 'GarageFinish', 'GarageType', 'FireplaceQu', 'MasVnrType']

In [393]:
preprocessor = ColumnTransformer([
    ('num', 'passthrough', numerical_features),
    ('cat', OneHotEncoder(), categorical_features)
])

In [394]:
pipeline_mixed = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

In [395]:
# X_mixed_transformed = preprocessor.fit_transform(X_mixed_train)

In [396]:
# X_mixed_transformed.shape

In [397]:
scores_mixed = cross_val_score(
    pipeline_mixed,
    X_mixed_train,
    y_mixed_train,
    cv=kf,
    scoring='neg_mean_absolute_error'
)

In [398]:
scores_mixed

array([-20812.47774173, -23090.59207415, -22234.1485784 , -22020.92649641,
       -19538.72482869])

In [399]:
positive_scores_mixed = -scores_mixed

In [400]:
positive_scores_mixed

array([20812.47774173, 23090.59207415, 22234.1485784 , 22020.92649641,
       19538.72482869])

In [401]:
positive_scores_mixed.mean()

np.float64(21539.373943875515)

In [402]:
positive_scores_mixed.std()

np.float64(1237.0818333558414)

In [403]:
# Linear Regression
# Mixed baseline
# 24 features
# Mean CV MAE ≈ $19000.17
# Std ≈ $1232.63

In [404]:
X_mixed_2 = dataset[['OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea', 'TotalBsmtSF',  'YearBuilt', 'YearRemodAdd', 'GarageYrBlt', 'MasVnrArea', 'Fireplaces', 'BsmtFinSF1', 'WoodDeckSF', 'LotArea', 'OverallCond', 'MSSubClass', 'EnclosedPorch', 'KitchenAbvGr', 'ExterQual', 'KitchenQual', 'BsmtQual', 'GarageFinish', 'GarageType', 'FireplaceQu', 'MasVnrType']]
y_mixed_2 = dataset['SalePrice']

In [405]:
X_mixed_train_2, X_mixed_test_2, y_mixed_train_2, y_mixed_test_2 = train_test_split(X_mixed_2, y_mixed_2, test_size=0.20, random_state=42)

In [406]:
numerical_features_2 = ['OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea', 'TotalBsmtSF',  'YearBuilt', 'YearRemodAdd', 'GarageYrBlt', 'MasVnrArea', 'Fireplaces', 'BsmtFinSF1', 'WoodDeckSF', 'LotArea', 'OverallCond', 'MSSubClass', 'EnclosedPorch', 'KitchenAbvGr']
categorical_features_2 = ['ExterQual', 'KitchenQual', 'BsmtQual', 'GarageFinish', 'GarageType', 'FireplaceQu', 'MasVnrType']

In [407]:
preprocessor_2 = ColumnTransformer([
    ('num', 'passthrough', numerical_features_2),
    ('cat', OneHotEncoder(), categorical_features_2)
])

In [408]:
pipeline_mixed_2 = Pipeline([
    ('preprocessor_2', preprocessor_2),
    ('model', LinearRegression())
])

In [409]:
scores_mixed_2 = cross_val_score(
    pipeline_mixed_2,
    X_mixed_train_2,
    y_mixed_train_2,
    cv=kf,
    scoring='neg_mean_absolute_error'
)

In [410]:
scores_mixed_2

array([-18361.13430324, -19554.58089335, -20702.38317117, -19347.9554425 ,
       -17034.80065002])

In [411]:
positive_scores_mixed_2 = -scores_mixed_2

In [412]:
positive_scores_mixed_2

array([18361.13430324, 19554.58089335, 20702.38317117, 19347.9554425 ,
       17034.80065002])

In [413]:
positive_scores_mixed_2.mean()

np.float64(19000.170892057755)

In [414]:
positive_scores_mixed_2.std()

np.float64(1232.631261680859)

In [ ]:
# Decision Tree Regressor
# Mixed baseline
# 24 features
# Mean CV MAE ≈ $26826.21
# Std ≈ $1491.49

In [418]:
# Experiment with DecisionTreeRegressor
X_mixed_3 = dataset[['OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea', 'TotalBsmtSF',  'YearBuilt', 'YearRemodAdd', 'GarageYrBlt', 'MasVnrArea', 'Fireplaces', 'BsmtFinSF1', 'WoodDeckSF', 'LotArea', 'OverallCond', 'MSSubClass', 'EnclosedPorch', 'KitchenAbvGr', 'ExterQual', 'KitchenQual', 'BsmtQual', 'GarageFinish', 'GarageType', 'FireplaceQu', 'MasVnrType']]
y_mixed_3 = dataset['SalePrice']

In [419]:
X_mixed_train_3, X_mixed_test_3, y_mixed_train_3, y_mixed_test_3 = train_test_split(X_mixed_3, y_mixed_3, test_size=0.20, random_state=42)

In [420]:
numerical_features_3 = ['OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea', 'TotalBsmtSF',  'YearBuilt', 'YearRemodAdd', 'GarageYrBlt', 'MasVnrArea', 'Fireplaces', 'BsmtFinSF1', 'WoodDeckSF', 'LotArea', 'OverallCond', 'MSSubClass', 'EnclosedPorch', 'KitchenAbvGr']
categorical_features_3 = ['ExterQual', 'KitchenQual', 'BsmtQual', 'GarageFinish', 'GarageType', 'FireplaceQu', 'MasVnrType']

In [421]:
preprocessor_3 = ColumnTransformer([
    ('num', 'passthrough', numerical_features_3),
    ('cat', OneHotEncoder(), categorical_features_3)
])

In [438]:
pipeline_mixed_3 = Pipeline([
    ('preprocessor_3', preprocessor_3),
    ('model', DecisionTreeRegressor())
])

In [439]:
scores_mixed_3 = cross_val_score(
    pipeline_mixed_3,
    X_mixed_train_3,
    y_mixed_train_3,
    cv=kf,
    scoring='neg_mean_absolute_error'
)

In [440]:
scores_mixed_3

array([-25180.86324786, -25490.25641026, -28564.05128205, -28821.81545064,
       -23666.68240343])

In [441]:
positive_scores_mixed_3 = -scores_mixed_3

In [442]:
positive_scores_mixed_3

array([25180.86324786, 25490.25641026, 28564.05128205, 28821.81545064,
       23666.68240343])

In [443]:
positive_scores_mixed_3.mean()

np.float64(26344.733758849638)

In [444]:
positive_scores_mixed_3.std()

np.float64(2015.836756508849)